# NGC4254: extracting conditional KTZ-compatible spiral source fields from the HII SFR map

This notebook takes the geometry and arm-profile model from the first two Python cells of `KTZ_validation.ipynb` and applies it to the `LOGSFR_SURFACE_DENSITY_HII` map in the current NGC4254 gas-bin product. Exactly `m=2,3,4` are compared. Each conditional fit records `m_arms`, negative `pitch_angle`, reference phase `Theta0`, exponential scale `h_R`, modulation `eta`, and harmonic arrays `harmonic_n`, `harmonic_g`, and `harmonic_alpha`.

Negative winding is imposed as the user-approved visual morphology prior. The result is a three-row conditional comparison for a lopsided, disturbed galaxy; no arm-number winner is declared. These models are low-dimensional descriptions rather than evidence for a rigid, stationary density wave, and diffusion, enrichment-age, clustering, and pattern-speed parameters remain outside this fit.


## 1. Model, conventions, and identifiability

The fitted linear source field is `lambda(R, phi) = lambda0_0 exp(-R/h_R) [1 + eta h(Theta)]`, where `Theta = (m/tan(pitch)) ln(R/R_ref) - m phi + Theta0`. The transverse arm profile is a harmonic series, `h(Theta) = sum g_n cos(n Theta + alpha_n)`.

Only the products `eta*g_n` are observable if every harmonic amplitude is free. We therefore fix `g_1 = 1` and `alpha_1 = 0`; the fundamental phase is carried by `Theta0`. This makes `eta` the fundamental modulation amplitude and makes the higher harmonic amplitudes and phases identifiable. Exactly `m=2,3,4` are compared, negative winding is imposed as the user-approved visual morphology prior, and no arm-number winner is declared under the explicit disc-coordinate convention below.


## 2. Imports and visible configuration

This cell contains every path, geometry assumption, optimizer range, and random seed used later. Brown et al. supplies the optical centre, inclination, and directed receding-side position angle. Equation 3 of Huang et al. (2026), https://academic.oup.com/mnras/article/549/3/stag1019/8698768, supplies the finite-thickness surface-density factor that is already applied upstream. Coordinate deprojection still uses `cos(i)`. The FITS WCS handles sky orientation, so the array is not manually flipped north-up/east-left before the tangent-plane rotation.

Exactly `m=2,3,4` are compared, negative winding is imposed as the user-approved visual morphology prior, and no arm-number winner is declared. The implementation contract is `leakage_controlled_negative_winding_ridge`: `conditional_ridge_search` will retain one conditional geometry per arm number in `ridge_geometry_table`, and completed real-data execution will emit `RIDGE_M234_REAL_COMPLETE`. Change these visible values here rather than hiding alternatives inside helper functions.


In [ ]:
from pathlib import Path
from time import perf_counter
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.coordinates import FK5, SkyCoord
from astropy.io import fits
from astropy.wcs import WCS
import astropy.units as u
from scipy.ndimage import gaussian_filter, gaussian_filter1d
from scipy.optimize import least_squares, minimize_scalar

ROOT = Path.cwd().resolve()
FITS_PATH = ROOT / "v3tk_v7.6.8/NGC4254/NGC4254_gas_bin_maps_further.fits"
BROWN_TABLE = ROOT / "Brown2021Table1.txt"
UPSTREAM_SFR_LOG = ROOT / "sfr_logs/NGC4254.log"
SFR_HDU = "LOGSFR_SURFACE_DENSITY_HII"
BIN_HDU = "BIN_ID"

DISTANCE_MPC = 16.5
Q0 = 0.2
SURFACE_DENSITY_ALREADY_CORRECTED = True
APPLY_BA_CORRECTION_IN_NOTEBOOK = False
R_REF_KPC = 1.0
M_COMPARE = np.array([2, 3, 4], dtype=int)
RIDGE_PITCH_GRID_DEG = np.arange(-45.0, -4.9, 1.0)
RIDGE_GUARD_BINS = 4
RIDGE_NULL_BLOCK_SEEDS = np.array([4254, 5254, 6254, 7254], dtype=int)
RIDGE_NULL_DRAWS_PER_BLOCK = 8
RIDGE_NULL_TOTAL_DRAWS = int(len(RIDGE_NULL_BLOCK_SEEDS) * RIDGE_NULL_DRAWS_PER_BLOCK)
RIDGE_CORE_WIDTH_KPC = 0.25
RIDGE_WIDTH_SENSITIVITY_KPC = np.array([0.18, 0.22, 0.25, 0.30, 0.35])
RIDGE_BROAD_RATIO = 3.0
RIDGE_N_PHASE = 360
RIDGE_N_SECTORS = 12
RIDGE_MIN_HELD_OUT_SECTORS = 10
RIDGE_SHORTLIST_PER_FAMILY = 5
LOGPOLAR_N_U = 120
LOGPOLAR_N_PHI = 360
LOGPOLAR_N_RADIAL_BANDS = 24
LOGPOLAR_SMOOTH_SIGMA = (0.8, 1.2)
LOGPOLAR_AZIMUTH_BROAD_SIGMA_BINS = 30.0
HARMONIC_N = np.array([1, 2, 3], dtype=int)
USE_ALL_VALID_HII_BINS = True
N_BOOTSTRAP = 24
N_BOOTSTRAP_SECTORS = 12
RNG_SEED = 4254
rng = np.random.default_rng(RNG_SEED)


def load_brown_geometry(path, galaxy="NGC 4254"):
    if not path.exists():
        raise FileNotFoundError(path)
    rows = [line.rstrip("\n") for line in path.read_text(encoding="utf-8").splitlines()
            if line.startswith(galaxy)]
    if len(rows) != 1:
        raise RuntimeError(f"Expected one {galaxy} row in {path}; found {len(rows)}")
    fields = rows[0].split("\t")
    if len(fields) < 6:
        raise ValueError(f"Malformed Brown table row: {rows[0]}")
    ra_match = re.fullmatch(r"(\d+)\^h(\d+)\^m(\d+)\.s(\d+)", fields[1])
    dec_match = re.fullmatch(r"([+-]?\d+)deg(\d+)'(\d+)\.\"(\d+)", fields[2])
    if ra_match is None or dec_match is None:
        raise ValueError(f"Unrecognized Brown coordinates: {fields[1:3]}")
    hh, mm, ss, sf = ra_match.groups()
    dd, dm, ds, df = dec_match.groups()
    sign = "-" if dd.startswith("-") else "+"
    dd_abs = dd.lstrip("+-")
    coordinate = SkyCoord(
        f"{hh}h{mm}m{ss}.{sf}s",
        f"{sign}{dd_abs}d{dm}m{ds}.{df}s",
        frame=FK5(equinox="J2000"),
    )
    return {
        "center": coordinate,
        "inclination_deg": float(fields[4]),
        "position_angle_deg": float(fields[5]),
        "source_row": rows[0],
    }


def finite_thickness_axis_ratio(inclination_deg, q0=Q0):
    cosine = np.cos(np.deg2rad(float(inclination_deg)))
    return float(np.sqrt((1.0 - q0**2) * cosine**2 + q0**2))


BROWN_GEOMETRY = load_brown_geometry(BROWN_TABLE)
CENTER = BROWN_GEOMETRY["center"]
INCLINATION_DEG = BROWN_GEOMETRY["inclination_deg"]
POSITION_ANGLE_DEG = BROWN_GEOMETRY["position_angle_deg"]
B_OVER_A = finite_thickness_axis_ratio(INCLINATION_DEG)
EXPECTED_CENTER = SkyCoord("12h18m49.68s", "+14d25m05.52s",
                           frame=FK5(equinox="J2000"))
assert CENTER.separation(EXPECTED_CENTER).to_value(u.arcsec) < 1e-6
assert np.isclose(INCLINATION_DEG, 39.0)
assert np.isclose(POSITION_ANGLE_DEG % 360.0, 243.0)
assert np.isclose(B_OVER_A, 0.787, atol=5e-4)
assert SURFACE_DENSITY_ALREADY_CORRECTED
assert not APPLY_BA_CORRECTION_IN_NOTEBOOK
upstream_text = UPSTREAM_SFR_LOG.read_text(encoding="utf-8")
upstream_matches = re.findall(
    r"Inclination correction ENABLED: applying b/a = ([0-9.]+)",
    upstream_text,
)
if not upstream_matches:
    raise RuntimeError(f"No enabled b/a correction record in {UPSTREAM_SFR_LOG}")
assert np.isclose(float(upstream_matches[-1]), B_OVER_A, atol=5e-4)
print(f"Brown catalog row: {BROWN_GEOMETRY['source_row']}")
print(f"Adopted FK5 J2000 centre: {CENTER.to_string('hmsdms')}")
print(f"Inclination={INCLINATION_DEG:.1f} deg; directed PA={POSITION_ANGLE_DEG:.1f} deg east of north")
print("BROWN_GEOMETRY_PASS")
print(f"Equation-3 finite-thickness factor b/a={B_OVER_A:.4f}; already applied upstream")
print("BA_FACTOR_PASS")
print(f"Confirmed upstream correction record: b/a={float(upstream_matches[-1]):.3f}")
print("UPSTREAM_BA_PASS")

plt.style.use("default")
plt.rcParams.update({"figure.dpi": 115, "font.size": 10})
warnings.filterwarnings("default")
print(f"Project root: {ROOT}")
print(f"Input product: {FITS_PATH}")


## 3. Read the two FITS maps and validate their shared geometry

The SFR map stores logarithmic surface density, while `BIN_ID` records the adaptive gas bin assigned to every image pixel. The loader copies both arrays before the FITS handle closes, verifies identical shapes, and constructs a celestial WCS from the SFR extension. No missing HII pixels are filled, and no science product is changed. The printed counts are live diagnostics rather than fixed sample assumptions.


In [ ]:
def load_maps(path, sfr_hdu=SFR_HDU, bin_hdu=BIN_HDU):
    'Read the log-SFR and bin-ID maps and return an independent celestial WCS.'
    if not path.exists():
        raise FileNotFoundError(path)
    with fits.open(path, memmap=True) as hdul:
        names = {hdu.name for hdu in hdul}
        missing = [name for name in (sfr_hdu, bin_hdu) if name not in names]
        if missing:
            raise KeyError(f"Missing required HDUs: {missing}")
        log_sfr = np.asarray(hdul[sfr_hdu].data, dtype=float).copy()
        bin_id = np.asarray(hdul[bin_hdu].data, dtype=float).copy()
        header = hdul[sfr_hdu].header.copy()
    if log_sfr.shape != bin_id.shape:
        raise ValueError(f"Map shape mismatch: {log_sfr.shape} versus {bin_id.shape}")
    wcs = WCS(header).celestial
    if not wcs.has_celestial:
        raise ValueError("Target HDU does not contain a celestial WCS")
    return log_sfr, bin_id, header, wcs


log_sfr_map, bin_id_map, sfr_header, celestial_wcs = load_maps(FITS_PATH)
wcs_reference = SkyCoord(
    celestial_wcs.wcs.crval[0] * u.deg,
    celestial_wcs.wcs.crval[1] * u.deg,
    frame=FK5(equinox="J2000"),
)
wcs_center_separation_arcsec = float(
    wcs_reference.separation(CENTER).to_value(u.arcsec))
assert wcs_center_separation_arcsec > 1.0
print(f"FITS CRVAL is a WCS projection reference, not the Brown optical centre; "
      f"separation={wcs_center_separation_arcsec:.3f} arcsec")
print("WCS_REFERENCE_NOT_CENTER_PASS")
finite_sfr = np.isfinite(log_sfr_map)
print(f"Map shape: {log_sfr_map.shape}")
print(f"Finite HII SFR pixels: {finite_sfr.sum():,}")
print(f"Finite log SFR range: {np.nanmin(log_sfr_map):.3f} to {np.nanmax(log_sfr_map):.3f}")


## 4. Collapse repeated pixels to one independent gas-bin record

Adaptive gas-bin values are copied to every member pixel, so treating all finite pixels as independent would produce false precision. This function builds one row per valid `BIN_ID`: the mean stored log SFR, pixel centroid, sky centroid, and member-pixel area. Later fits use the member area as an integration weight for independent-bin KTZ fitting and leave-one-sector-out morphology stability.


In [ ]:
def build_bin_catalog(log_sfr, bin_id, wcs):
    'Return one centroid and one SFR measurement per valid adaptive gas bin.'
    valid = np.isfinite(log_sfr) & np.isfinite(bin_id) & (bin_id >= 0)
    yy, xx = np.nonzero(valid)
    ids = bin_id[valid].astype(np.int64)
    values = log_sfr[valid]
    unique_ids, inverse = np.unique(ids, return_inverse=True)
    area_pix = np.bincount(inverse).astype(float)
    x_centroid = np.bincount(inverse, weights=xx) / area_pix
    y_centroid = np.bincount(inverse, weights=yy) / area_pix
    mean_log_sfr = np.bincount(inverse, weights=values) / area_pix
    ra_deg, dec_deg = wcs.pixel_to_world_values(x_centroid, y_centroid)
    table = pd.DataFrame({
        "bin_id": unique_ids,
        "x_pix": x_centroid,
        "y_pix": y_centroid,
        "ra_deg": ra_deg,
        "dec_deg": dec_deg,
        "area_pix": area_pix,
        "log_sfr": mean_log_sfr,
        "sfr_linear": np.power(10.0, mean_log_sfr),
    })
    if len(table) < 100:
        raise ValueError(f"Only {len(table)} valid HII bins; spiral fitting is not supported")
    return table, valid


bin_table, valid_hii_pixels = build_bin_catalog(log_sfr_map, bin_id_map, celestial_wcs)
assert bin_table["bin_id"].is_unique
assert np.all(bin_table["area_pix"] > 0)
assert np.all(np.isfinite(bin_table["sfr_linear"]))
print(f"Independent valid HII gas bins: {len(bin_table):,}")
print(f"Median represented area: {bin_table['area_pix'].median():.0f} image pixels")
display(bin_table.head())


## 5. Inspect the observed map and the adopted sky-plane geometry

Before deprojection, this plot checks that the WCS, optical centre, and position-angle convention are mutually sensible. The white line is the adopted major-axis direction and the cyan line is the perpendicular projected minor axis. A sign or axis-order error here would change the spiral winding and pitch, so this visualization is a required geometry check rather than decorative plotting.


In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 7.2))
vmin, vmax = np.nanpercentile(log_sfr_map[finite_sfr], [2, 98])
image = ax.imshow(log_sfr_map, origin="lower", cmap="magma", vmin=vmin, vmax=vmax)
cx, cy = celestial_wcs.world_to_pixel(CENTER)
ax.scatter(cx, cy, marker="+", s=140, linewidth=2.0, color="lime", label="adopted centre")
length_pix = 260.0
pa = np.deg2rad(POSITION_ANGLE_DEG)
# Image x increases approximately east-to-west because CDELT1 is negative; draw axes via sky offsets.
for angle_deg, color, label in [(POSITION_ANGLE_DEG, "white", "major axis"),
                                (POSITION_ANGLE_DEG + 90.0, "cyan", "minor axis")]:
    angle = np.deg2rad(angle_deg)
    dra = np.sin(angle) * 35.0 * u.arcsec
    ddec = np.cos(angle) * 35.0 * u.arcsec
    end1 = CENTER.directional_offset_by(np.arctan2(dra.value, ddec.value) * u.rad,
                                        np.hypot(dra.value, ddec.value) * u.arcsec)
    end2 = CENTER.directional_offset_by((np.arctan2(dra.value, ddec.value) + np.pi) * u.rad,
                                        np.hypot(dra.value, ddec.value) * u.arcsec)
    x1, y1 = celestial_wcs.world_to_pixel(end1)
    x2, y2 = celestial_wcs.world_to_pixel(end2)
    ax.plot([x1, x2], [y1, y2], color=color, lw=1.5, label=label)
ax.set_xlim(0, log_sfr_map.shape[1]-1)
ax.set_ylim(0, log_sfr_map.shape[0]-1)
ax.set_aspect("equal")
ax.set_xlabel("image x (pixel; celestial WCS retained in FITS)")
ax.set_ylabel("image y (pixel)")
ax.set_title("NGC4254 HII log SFR surface density and adopted geometry")
ax.legend(loc="upper right", fontsize=8)
fig.colorbar(image, ax=ax, pad=0.02, label=r"log $\Sigma_{\rm SFR}$")
plt.show()


## 6. Deproject gas-bin centroids into the galaxy plane

Deprojection asks where each sky position would lie if the inclined disc were viewed face-on. Astropy first calculates tangent-plane offsets from the adopted centre: `east` is positive toward increasing right ascension and `north` is positive toward increasing declination. Multiplying the angular offsets by the 16.5 Mpc angular scale converts them to kpc.

The adopted position angle PA is measured east of north. We rotate the sky offsets into the projected galaxy axes using `major = east*sin(PA) + north*cos(PA)` and `projected_minor = -east*cos(PA) + north*sin(PA)`. Inclination shortens only the apparent minor axis, so the face-on coordinate is `minor = projected_minor/cos(inclination)`. Radius and azimuth are then `R = sqrt(major^2 + minor^2)` and `phi = atan2(minor, major)`.

For projecting a fitted skeleton back to the image, the algebra is reversed: `east = major*sin(PA) - projected_minor*cos(PA)` and `north = major*cos(PA) + projected_minor*sin(PA)`, with `projected_minor = minor*cos(inclination)`. The FITS WCS converts those east/north offsets back to image pixels. The centre, inclination, and PA are fixed rather than fitted from the patchy HII map because they are strongly degenerate with spiral phase, pitch, and winding sign.


In [ ]:
ARCSEC_TO_KPC = DISTANCE_MPC * 1.0e3 / 206265.0


def deproject_sky(ra_deg, dec_deg):
    'Convert FK5 J2000 sky coordinates to deprojected major/minor coordinates in kpc.'
    coords = SkyCoord(np.asarray(ra_deg) * u.deg,
                      np.asarray(dec_deg) * u.deg,
                      frame=FK5(equinox="J2000"))
    dlon, dlat = CENTER.spherical_offsets_to(coords)
    east_kpc = dlon.to_value(u.arcsec) * ARCSEC_TO_KPC
    north_kpc = dlat.to_value(u.arcsec) * ARCSEC_TO_KPC
    pa_rad = np.deg2rad(POSITION_ANGLE_DEG)
    inc_rad = np.deg2rad(INCLINATION_DEG)
    major = east_kpc * np.sin(pa_rad) + north_kpc * np.cos(pa_rad)
    minor_projected = -east_kpc * np.cos(pa_rad) + north_kpc * np.sin(pa_rad)
    minor = minor_projected / np.cos(inc_rad)
    radius = np.hypot(major, minor)
    azimuth = np.arctan2(minor, major)
    return major, minor, radius, azimuth


# A direct unit check: the adopted sky centre must map to the disc origin.
origin = deproject_sky([CENTER.ra.deg], [CENTER.dec.deg])
assert np.allclose([origin[0][0], origin[1][0]], 0.0, atol=1.0e-10)

values = deproject_sky(bin_table["ra_deg"], bin_table["dec_deg"])
bin_table[["x_disc_kpc", "y_disc_kpc", "radius_kpc", "azimuth_rad"]] = np.column_stack(values)
# Keep every finite HII bin. The logarithmic spiral is undefined only at exactly R=0;
# no valid bin centroid lies exactly at the adopted centre in this product.
fit_mask = np.isfinite(bin_table["radius_kpc"]) & (bin_table["radius_kpc"] > 0.0)
fit_table = bin_table.loc[fit_mask].reset_index(drop=True)
R_IN_KPC = float(fit_table["radius_kpc"].min())
R_OUT_KPC = float(fit_table["radius_kpc"].max())
assert USE_ALL_VALID_HII_BINS
assert len(fit_table) == len(bin_table)
assert int(fit_table["area_pix"].sum()) == int(valid_hii_pixels.sum())
print(f"1 arcsec = {ARCSEC_TO_KPC:.4f} kpc at {DISTANCE_MPC:.1f} Mpc")
print(f"Fitting radius: {R_IN_KPC:.4f} to {R_OUT_KPC:.4f} kpc; no artificial centre or outer cut")
print(f"Retained bins: {len(fit_table):,} / {len(bin_table):,}")
print(f"Represented valid HII pixels: {int(fit_table['area_pix'].sum()):,} / {int(valid_hii_pixels.sum()):,}")
print("FIT_DOMAIN_ALL_VALID_HII_PASS")

fig, ax = plt.subplots(figsize=(7.5, 7.0))
sc = ax.scatter(fit_table["x_disc_kpc"], fit_table["y_disc_kpc"],
                c=fit_table["log_sfr"], s=np.clip(fit_table["area_pix"]**0.5, 2, 15),
                cmap="magma", vmin=vmin, vmax=vmax, alpha=0.75, linewidth=0)
ax.set_aspect("equal")
ax.set_xlabel("disc major-axis x (kpc)")
ax.set_ylabel("deprojected minor-axis y (kpc)")
ax.set_title("Deprojected independent HII gas bins")
fig.colorbar(sc, ax=ax, label=r"log $\Sigma_{\rm SFR}$")
plt.show()


### 6a. Build all-pixel support for the deprojected ridge morphology

The ridge map needs spatial coverage at image-pixel resolution, so every finite
HII pixel supplies morphology support after WCS deprojection. This is deliberately
separate from the unique-bin catalogue: unique bins remain the independent fitting
records, while repeated member pixels describe where each measurement covers the
disc. The geometry below therefore uses all available HII spaxels, requires only
finite values and the mathematical R > 0 guard, and applies no artificial centre
cut.


In [ ]:
def build_hii_pixel_geometry(log_sfr, valid_mask, wcs):
    yy, xx = np.nonzero(valid_mask)
    ra_deg, dec_deg = wcs.pixel_to_world_values(xx, yy)
    major, minor, radius, azimuth = deproject_sky(ra_deg, dec_deg)
    table = pd.DataFrame({
        "x_pix": xx.astype(float),
        "y_pix": yy.astype(float),
        "x_disc_kpc": major,
        "y_disc_kpc": minor,
        "radius_kpc": radius,
        "azimuth_rad": azimuth,
        "log_sfr": log_sfr[valid_mask],
    })
    keep = np.isfinite(table).all(axis=1) & (table["radius_kpc"] > 0)
    result = table.loc[keep].reset_index(drop=True)
    if len(result) != int(valid_mask.sum()):
        raise RuntimeError(f"Lost {int(valid_mask.sum()) - len(result)} valid HII pixels")
    return result


hii_pixels = build_hii_pixel_geometry(log_sfr_map, valid_hii_pixels, celestial_wcs)
assert len(hii_pixels) == int(valid_hii_pixels.sum())
RIDGE_R_IN_KPC = float(hii_pixels["radius_kpc"].min())
RIDGE_R_OUT_KPC = float(hii_pixels["radius_kpc"].max())
RIDGE_PIVOT_RADIUS_KPC = float(np.exp(
    np.median(np.log(hii_pixels["radius_kpc"].to_numpy(float)))))
print(f"Ridge morphology pixels: {len(hii_pixels):,} / {int(valid_hii_pixels.sum()):,}")
print(f"Ridge pixel radius: {RIDGE_R_IN_KPC:.4f} to {RIDGE_R_OUT_KPC:.4f} kpc; "
      f"phase pivot={RIDGE_PIVOT_RADIUS_KPC:.3f} kpc")
print("RIDGE_PIXEL_DOMAIN_PASS")


## 7. Fit the axisymmetric background and build the log-polar ridge field

The KTZ source field separates into an exponential radial background and a
fractional spiral modulation. Here h_R is the e-folding length of the
axisymmetric SFR source field: increasing radius by h_R lowers that background
by a factor of e. It is not an arm width, pitch angle, optical radius, or
automatically a stellar-disc scale length. The robust fit uses independent bins
and their represented areas, while the morphology field uses every finite HII
pixel.

A live falsification of the earlier coverage rule showed why the mask handling
matters: a global 0.05 * max(coverage) threshold retained only about 14% of
log-polar cells and removed every row below roughly 2.5 kpc, even though those
central pixels entered the histogram. Equal-count quantile rows plus an absolute
local coverage floor retain those pixels, normalize only where local support
exists, and do not create a centre hole. Signed residuals and their negative
flanks are preserved before broad-azimuth structure is removed.


In [ ]:
def fit_radial_exponential(radius, sfr_linear, area_weight):
    'Robustly fit lambda0*exp(-R/h_R) in log space.'
    radius = np.asarray(radius, dtype=float)
    sfr_linear = np.asarray(sfr_linear, dtype=float)
    weights = np.sqrt(np.asarray(area_weight, dtype=float))
    valid = (np.isfinite(radius) & np.isfinite(sfr_linear) &
             np.isfinite(weights) & (sfr_linear > 0) & (weights > 0))
    r = radius[valid]
    y = np.log(sfr_linear[valid])
    w = weights[valid] / np.nanmedian(weights[valid])
    initial = np.array([np.nanmedian(y), np.log(3.0)])

    def residual(theta):
        log_lambda0, log_h = theta
        return w * (y - (log_lambda0 - r / np.exp(log_h)))

    result = least_squares(residual, initial, loss="soft_l1", f_scale=0.3,
                           bounds=([-50.0, np.log(0.2)], [50.0, np.log(30.0)]))
    if not result.success:
        raise RuntimeError(result.message)
    return {"lambda0_0": float(np.exp(result.x[0])),
            "h_R": float(np.exp(result.x[1])),
            "cost": float(2.0 * result.cost), "success": True}


# Synthetic unit check of the radial fitter.
r_test = np.linspace(0.5, 8.0, 200)
radial_test = fit_radial_exponential(r_test, 0.025*np.exp(-r_test/3.4), np.ones_like(r_test))
assert abs(radial_test["h_R"] - 3.4) < 0.02

radial_fit = fit_radial_exponential(fit_table["radius_kpc"],
                                    fit_table["sfr_linear"], fit_table["area_pix"])
radial_fit["gradient_dex_per_kpc"] = float(
    -1.0 / (np.log(10.0) * radial_fit["h_R"]))
print(f"Axisymmetric SFR background h_R={radial_fit['h_R']:.3f} kpc; "
      f"gradient={radial_fit['gradient_dex_per_kpc']:.4f} dex/kpc")



def preprocess_logpolar_raw(
        raw_state, allowed_phi=None, include_local=True,
        smooth_sigma=LOGPOLAR_SMOOTH_SIGMA,
        broad_sigma_phi=LOGPOLAR_AZIMUTH_BROAD_SIGMA_BINS):
    """Validate, mask, and smooth a raw log-polar histogram."""
    required = (
        "raw_weighted_sum", "raw_coverage", "u", "phi",
        "u_edges", "phi_edges",
    )
    missing = [key for key in required if key not in raw_state]
    if missing:
        raise ValueError(f"Raw log-polar state is missing keys: {missing}")

    raw_weighted_sum = np.array(
        raw_state["raw_weighted_sum"], dtype=float, copy=True)
    raw_coverage = np.array(
        raw_state["raw_coverage"], dtype=float, copy=True)
    if raw_weighted_sum.ndim != 2 or raw_coverage.ndim != 2:
        raise ValueError("Raw weighted sum and coverage must be 2-D")
    if raw_weighted_sum.shape != raw_coverage.shape:
        raise ValueError("Raw weighted sum and coverage shapes differ")
    if not np.isfinite(raw_weighted_sum).all():
        raise ValueError("Raw weighted sum must be finite")
    if not np.isfinite(raw_coverage).all() or np.any(raw_coverage < 0.0):
        raise ValueError("Raw coverage must be finite and non-negative")

    u = np.array(raw_state["u"], dtype=float, copy=True)
    phi = np.array(raw_state["phi"], dtype=float, copy=True)
    u_edges = np.array(raw_state["u_edges"], dtype=float, copy=True)
    phi_edges = np.array(raw_state["phi_edges"], dtype=float, copy=True)
    n_u, n_phi = raw_weighted_sum.shape
    if u.ndim != 1 or u.size != n_u:
        raise ValueError("u must be a 1-D coordinate matching the radial axis")
    if phi.ndim != 1 or phi.size != n_phi:
        raise ValueError("phi must be a 1-D coordinate matching the azimuth axis")
    if u_edges.ndim != 1 or u_edges.size != n_u + 1:
        raise ValueError("u_edges must have one more entry than the radial axis")
    if phi_edges.ndim != 1 or phi_edges.size != n_phi + 1:
        raise ValueError(
            "phi_edges must have one more entry than the azimuth axis")
    for label, values in (
            ("u", u), ("phi", phi),
            ("u_edges", u_edges), ("phi_edges", phi_edges)):
        if not np.isfinite(values).all():
            raise ValueError(f"{label} must be finite")
    if np.any(np.diff(u_edges) <= 0.0):
        raise ValueError("u_edges must be strictly increasing")
    if np.any(np.diff(phi_edges) <= 0.0):
        raise ValueError("phi_edges must be strictly increasing")

    allowed = None
    if allowed_phi is not None:
        allowed = np.array(allowed_phi, dtype=bool, copy=True)
        if allowed.ndim != 1 or allowed.size != n_phi:
            raise ValueError("allowed_phi must match the log-polar azimuth axis")
        raw_weighted_sum[:, ~allowed] = 0.0
        raw_coverage[:, ~allowed] = 0.0

    numerator = gaussian_filter(
        raw_weighted_sum, smooth_sigma, mode=("nearest", "wrap"))
    denominator = gaussian_filter(
        raw_coverage, smooth_sigma, mode=("nearest", "wrap"))
    valid = (
        (denominator > 0.05)
        & np.isfinite(numerator)
        & np.isfinite(denominator)
    )
    radial_residual = np.divide(
        numerator, denominator, out=np.full_like(numerator, np.nan),
        where=valid)

    broad_azimuthal = np.full_like(radial_residual, np.nan)
    local_ridge = np.full_like(radial_residual, np.nan)
    if include_local:
        broad_numerator = gaussian_filter1d(
            np.where(valid, radial_residual * denominator, 0.0),
            broad_sigma_phi, axis=1, mode="wrap")
        broad_denominator = gaussian_filter1d(
            np.where(valid, denominator, 0.0),
            broad_sigma_phi, axis=1, mode="wrap")
        broad_azimuthal = np.divide(
            broad_numerator, broad_denominator,
            out=np.full_like(broad_numerator, np.nan),
            where=broad_denominator > 0.0)
        local_ridge[valid] = (
            radial_residual[valid] - broad_azimuthal[valid])

    if allowed is not None:
        disallowed = ~allowed
        valid[:, disallowed] = False
        denominator[:, disallowed] = 0.0
        radial_residual[:, disallowed] = np.nan
        local_ridge[:, disallowed] = np.nan
        broad_azimuthal[:, disallowed] = np.nan

    return {
        "radial_residual": radial_residual,
        "local_ridge": local_ridge,
        "coverage": denominator,
        "valid": valid,
        "u": u,
        "phi": phi,
        "u_edges": u_edges,
        "phi_edges": phi_edges,
        "broad_azimuthal": broad_azimuthal,
    }


def build_log_polar_contrast(
        pixel_table, radial_parameters,
        n_u=LOGPOLAR_N_U, n_phi=LOGPOLAR_N_PHI):
    """Build unsmoothed log-polar sums, then derive display fields."""
    n_u = int(n_u)
    n_phi = int(n_phi)
    if n_u < 1 or n_phi < 1:
        raise ValueError("Log-polar dimensions must be positive")

    radius = pixel_table["radius_kpc"].to_numpy(float)
    azimuth = pixel_table["azimuth_rad"].to_numpy(float)
    observed_log = pixel_table["log_sfr"].to_numpy(float)
    if not (radius.ndim == azimuth.ndim == observed_log.ndim == 1):
        raise ValueError("Log-polar pixel columns must be one-dimensional")
    if not (radius.size == azimuth.size == observed_log.size) or radius.size == 0:
        raise ValueError("Log-polar pixel columns must have equal nonzero length")
    if not np.isfinite(radius).all() or np.any(radius <= 0.0):
        raise ValueError("Log-polar radii must be finite and positive")
    if not np.isfinite(azimuth).all():
        raise ValueError("Log-polar azimuths must be finite")
    if not np.isfinite(observed_log).all():
        raise ValueError("Log-polar SFR values must be finite")

    lambda0_0 = float(radial_parameters["lambda0_0"])
    h_r = float(radial_parameters["h_R"])
    if not np.isfinite(lambda0_0) or lambda0_0 <= 0.0:
        raise ValueError("lambda0_0 must be finite and positive")
    if not np.isfinite(h_r) or h_r <= 0.0:
        raise ValueError("h_R must be finite and positive")

    background_log = (
        np.log10(lambda0_0) - radius / (np.log(10.0) * h_r))
    signed_residual = observed_log - background_log
    u = np.log(radius / R_REF_KPC)
    if not np.isfinite(u).all() or not np.isfinite(signed_residual).all():
        raise ValueError("Derived log-polar coordinates must be finite")

    # Validate repeated quantiles before nextafter expands the endpoints.
    quantile_edges = np.quantile(u, np.linspace(0.0, 1.0, n_u + 1))
    if np.any(np.diff(quantile_edges) <= 0.0):
        raise ValueError(
            "Quantile log-radius edges are not strictly increasing")
    u_edges = np.array(quantile_edges, copy=True)
    u_edges[0] = np.nextafter(u_edges[0], -np.inf)
    u_edges[-1] = np.nextafter(u_edges[-1], np.inf)
    phi_edges = np.linspace(-np.pi, np.pi, n_phi + 1)

    raw_weighted_sum, _, _ = np.histogram2d(
        u, azimuth, bins=(u_edges, phi_edges), weights=signed_residual)
    raw_coverage, _, _ = np.histogram2d(
        u, azimuth, bins=(u_edges, phi_edges))
    row_index = np.clip(
        np.searchsorted(u_edges, u, side="right") - 1, 0, n_u - 1)
    row_count = np.bincount(row_index, minlength=n_u)
    if np.any(row_count <= 0):
        raise ValueError("Every log-radius row must contain at least one pixel")
    u_centres = (
        np.bincount(row_index, weights=u, minlength=n_u) / row_count)
    if not np.isfinite(u_centres).all():
        raise ValueError("Log-radius row centres must be finite")

    raw_state = {
        "raw_weighted_sum": raw_weighted_sum,
        "raw_coverage": raw_coverage,
        "u": u_centres,
        "phi": 0.5 * (phi_edges[:-1] + phi_edges[1:]),
        "u_edges": u_edges,
        "phi_edges": phi_edges,
    }
    return raw_state, preprocess_logpolar_raw(raw_state, include_local=True)


logpolar_raw, logpolar = build_log_polar_contrast(hii_pixels, radial_fit)
assert np.isfinite(logpolar["radial_residual"][logpolar["valid"]]).all()
assert np.isfinite(logpolar["local_ridge"][logpolar["valid"]]).all()
assert int(logpolar_raw["raw_coverage"].sum()) == len(hii_pixels)
assert len(hii_pixels) == int(valid_hii_pixels.sum())

fig, axes = plt.subplots(1, 2, figsize=(14.0, 5.2), constrained_layout=True,
                         sharex=True, sharey=True)
for ax, field, title in zip(
        axes, ["radial_residual", "local_ridge"],
        ["signed residual after radial background",
         "local ridge field after broad-azimuth removal"]):
    image = ax.pcolormesh(
        logpolar["phi_edges"], logpolar["u_edges"], logpolar[field],
        shading="auto", cmap="coolwarm")
    ax.set(xlabel=r"disc azimuth $\phi$ (rad)",
           ylabel=r"$\ln(R/R_{\rm ref})$", title=title)
    fig.colorbar(image, ax=ax, label="residual (dex)")
plt.show()


radial_model = radial_fit["lambda0_0"] * np.exp(-fit_table["radius_kpc"].to_numpy()/radial_fit["h_R"])
fit_table["radial_model"] = radial_model
fit_table["q"] = fit_table["sfr_linear"].to_numpy()/radial_model - 1.0
print(radial_fit)

radial_edges = np.linspace(R_IN_KPC, R_OUT_KPC, 28)
radial_index = np.digitize(fit_table["radius_kpc"], radial_edges) - 1
summary_rows = []
for idx in range(len(radial_edges)-1):
    use = radial_index == idx
    if use.sum() >= 10:
        summary_rows.append((np.median(fit_table.loc[use, "radius_kpc"]),
                             np.median(fit_table.loc[use, "log_sfr"]), use.sum()))
radial_summary = pd.DataFrame(summary_rows, columns=["radius_kpc", "median_log_sfr", "n"])

fig, ax = plt.subplots(figsize=(8.2, 5.2))
ax.scatter(fit_table["radius_kpc"], fit_table["log_sfr"], s=2, alpha=0.06, color="tab:blue")
ax.plot(radial_summary["radius_kpc"], radial_summary["median_log_sfr"], "o", color="black", label="radial medians")
r_line = np.linspace(R_IN_KPC, R_OUT_KPC, 400)
ax.plot(r_line, np.log10(radial_fit["lambda0_0"]*np.exp(-r_line/radial_fit["h_R"])),
        color="tab:red", lw=2.2, label=fr"fit: $h_R={radial_fit['h_R']:.2f}$ kpc")
ax.set(xlabel="deprojected radius (kpc)", ylabel=r"log $\Sigma_{\rm SFR}$",
       title="Axisymmetric exponential background")
ax.legend()
plt.show()


## 8. Define the KTZ spiral phase and select arm number, winding, and phase from SFR ridges

The signed logarithmic-spiral phase maps a coherent ridge into a nearly constant log-polar phase. Its physical transverse distance obeys

`d_perp approximately R * abs(sin(pitch)) * abs(Theta) / m`

so a constant physical corridor must use the radius-dependent width

`sigma_Theta(R) = m * width_kpc / (R * abs(sin(pitch))).`

The rejected implementation used one phase width fixed at `R_ref`: at `R=5` and `10` kpc a nominal 0.35-kpc corridor broadened to approximately 1.75 and 3.5 kpc, respectively, rewarding lopsidedness rather than a narrow arm ridge. The replacement measures a narrow phase-smoothed mean minus a broad local-flank mean in approximately equal-count radial bands. Dividing weighted signal by coverage makes each branch a mean, not a sum, so increasing `m_arms` does not win merely by contributing more repeated branches.

Every `m/sign/pitch` candidate is initially screened on the full map. Exactly five pitches per `m` and winding sign enter screened azimuth-sector validation: each held-out sector is scored at a phase learned from the remaining sectors. This is screened sector validation, not fully nested cross-validation and not an unbiased posterior. The validated ridge score freezes `m_arms`, signed `pitch_angle`, and `Theta0`; no later optimizer is allowed to refine or relabel that geometry.

In [ ]:

def sector_columns(phi, n_sectors=RIDGE_N_SECTORS):
    """Map log-polar azimuth columns onto clipped sector indices."""
    phi = np.asarray(phi, dtype=float)
    sector = np.floor(
        (phi + np.pi) / (2.0 * np.pi) * int(n_sectors)).astype(int)
    return np.clip(sector, 0, int(n_sectors) - 1)


def circular_dilate(mask, guard_bins):
    """Dilate a one-dimensional circular mask by guard_bins columns."""
    mask = np.asarray(mask, dtype=bool)
    guard_bins = int(guard_bins)
    if mask.ndim != 1 or guard_bins < 0:
        raise ValueError("Circular dilation needs a 1D mask and non-negative guard")
    shifted = [np.roll(mask, shift)
               for shift in range(-guard_bins, guard_bins + 1)]
    return np.logical_or.reduce(shifted)


def circular_erode(mask, guard_bins):
    """Erode a one-dimensional circular mask by guard_bins columns."""
    mask = np.asarray(mask, dtype=bool)
    guard_bins = int(guard_bins)
    if mask.ndim != 1 or guard_bins < 0:
        raise ValueError("Circular erosion needs a 1D mask and non-negative guard")
    shifted = [np.roll(mask, shift)
               for shift in range(-guard_bins, guard_bins + 1)]
    return np.logical_and.reduce(shifted)


def logpolar_fold_masks(
        phi, held_sector, n_sectors=RIDGE_N_SECTORS,
        guard_bins=RIDGE_GUARD_BINS):
    """Return held, guarded-training, and eroded-test azimuth masks."""
    held_sector = int(held_sector)
    n_sectors = int(n_sectors)
    if held_sector < 0 or held_sector >= n_sectors:
        raise ValueError("held_sector lies outside the requested sector range")
    held_columns = sector_columns(phi, n_sectors) == held_sector
    train_allowed = ~circular_dilate(held_columns, guard_bins)
    test_allowed = circular_erode(held_columns, guard_bins)
    if np.any(train_allowed & test_allowed):
        raise RuntimeError("Training and test azimuth masks overlap")
    return {
        "held_columns": held_columns,
        "train_allowed": train_allowed,
        "test_allowed": test_allowed,
    }


def build_sector_fold_maps(
        raw_state, n_sectors=RIDGE_N_SECTORS,
        guard_bins=RIDGE_GUARD_BINS):
    """Split raw azimuth columns before independently smoothing each fold."""
    folds = []
    for held_sector in range(int(n_sectors)):
        masks = logpolar_fold_masks(
            raw_state["phi"], held_sector, n_sectors, guard_bins)
        train = preprocess_logpolar_raw(
            raw_state, allowed_phi=masks["train_allowed"],
            include_local=False)
        test = preprocess_logpolar_raw(
            raw_state, allowed_phi=masks["test_allowed"],
            include_local=False)
        folds.append({
            "held_sector": held_sector,
            "held_columns": masks["held_columns"],
            "train_allowed": masks["train_allowed"],
            "test_allowed": masks["test_allowed"],
            "train": train,
            "test": test,
        })
    return folds


def run_adversarial_leakage_check(
        n_u=12, n_phi=48, n_sectors=4,
        held_sector=1, guard_bins=2):
    """Test held-signal exclusion and a known nonzero training control."""
    n_u = int(n_u)
    n_phi = int(n_phi)
    u_edges = np.linspace(-1.0, 1.0, n_u + 1)
    phi_edges = np.linspace(-np.pi, np.pi, n_phi + 1)
    u_centres = 0.5 * (u_edges[:-1] + u_edges[1:])
    phi_centres = 0.5 * (phi_edges[:-1] + phi_edges[1:])
    masks = logpolar_fold_masks(
        phi_centres, held_sector, n_sectors, guard_bins)
    held_columns = masks["held_columns"]
    train_allowed = masks["train_allowed"]
    raw_coverage = np.full((n_u, n_phi), 20.0)

    def make_raw_state(weighted_sum):
        return {
            "raw_weighted_sum": np.array(weighted_sum, copy=True),
            "raw_coverage": np.array(raw_coverage, copy=True),
            "u": u_centres,
            "phi": phi_centres,
            "u_edges": u_edges,
            "phi_edges": phi_edges,
        }

    # First probe: all signal is confined to the held sector. It must leave
    # no residual in the guarded training field.
    held_only_weighted = np.zeros_like(raw_coverage)
    held_only_weighted[:, held_columns] = raw_coverage[:, held_columns]
    held_only_state = make_raw_state(held_only_weighted)
    fold = build_sector_fold_maps(
        held_only_state, n_sectors=n_sectors,
        guard_bins=guard_bins)[int(held_sector)]
    train = fold["train"]
    raw_training_signal_l1 = float(np.sum(np.abs(
        held_only_weighted[:, train_allowed])))
    training_values = train["radial_residual"][train["valid"]]
    max_abs_training_residual = (
        float(np.max(np.abs(training_values)))
        if training_values.size else 0.0)

    # Independent reference: physically excise every disallowed raw column,
    # then smooth without passing an allowed mask.
    reference_state = {
        key: np.array(value, copy=True)
        for key, value in held_only_state.items()
    }
    reference_state["raw_weighted_sum"][:, ~train_allowed] = 0.0
    reference_state["raw_coverage"][:, ~train_allowed] = 0.0
    reference = preprocess_logpolar_raw(
        reference_state, allowed_phi=None, include_local=False)
    training_support = np.broadcast_to(
        train_allowed[None, :], train["valid"].shape)
    common_valid = training_support & train["valid"] & reference["valid"]
    train_valid_count = int(np.count_nonzero(common_valid))
    max_abs_residual_difference = (
        float(np.max(np.abs(
            train["radial_residual"][common_valid]
            - reference["radial_residual"][common_valid])))
        if train_valid_count else np.inf)
    max_abs_coverage_difference = float(np.max(np.abs(
        train["coverage"][:, train_allowed]
        - reference["coverage"][:, train_allowed])))

    # Second probe: a constant, independently known training residual ensures
    # that an implementation cannot pass by discarding all weighted data.
    expected_training_residual = 0.25
    known_weighted = np.zeros_like(raw_coverage)
    known_weighted[:, train_allowed] = (
        expected_training_residual * raw_coverage[:, train_allowed])
    known_weighted[:, held_columns] = raw_coverage[:, held_columns]
    known_state = make_raw_state(known_weighted)
    known_train = build_sector_fold_maps(
        known_state, n_sectors=n_sectors,
        guard_bins=guard_bins)[int(held_sector)]["train"]
    known_values = known_train["radial_residual"][known_train["valid"]]
    known_train_valid_count = int(known_values.size)
    mean_known_training_residual = (
        float(np.mean(known_values)) if known_values.size else np.nan)
    max_abs_known_training_error = (
        float(np.max(np.abs(
            known_values - expected_training_residual)))
        if known_values.size else np.inf)

    return {
        "raw_training_signal_l1": raw_training_signal_l1,
        "max_abs_training_residual": max_abs_training_residual,
        "train_valid_count": train_valid_count,
        "max_abs_residual_difference": max_abs_residual_difference,
        "max_abs_coverage_difference": max_abs_coverage_difference,
        "known_train_valid_count": known_train_valid_count,
        "expected_training_residual": expected_training_residual,
        "mean_known_training_residual": mean_known_training_residual,
        "max_abs_known_training_error": max_abs_known_training_error,
    }


adversarial_leakage = run_adversarial_leakage_check()
assert adversarial_leakage["raw_training_signal_l1"] == 0.0
assert adversarial_leakage["max_abs_training_residual"] <= 1.0e-14
assert adversarial_leakage["train_valid_count"] > 0
assert adversarial_leakage["max_abs_residual_difference"] <= 1.0e-14
assert adversarial_leakage["max_abs_coverage_difference"] <= 1.0e-14
assert adversarial_leakage["known_train_valid_count"] > 0
assert abs(adversarial_leakage["mean_known_training_residual"]
           - adversarial_leakage["expected_training_residual"]) <= 1.0e-14
assert adversarial_leakage["max_abs_known_training_error"] <= 1.0e-14
print("RIDGE_LEAKAGE_GUARD_PASS")


def wrap_angle(angle):
    return np.angle(np.exp(1j*np.asarray(angle)))


def spiral_phase(radius, azimuth, m_arms, pitch_angle, theta0=0.0):
    pitch = np.deg2rad(pitch_angle)
    return ((m_arms/np.tan(pitch))*np.log(np.asarray(radius)/R_REF_KPC)
            - m_arms*np.asarray(azimuth) + theta0)


def arm_profile(theta, harmonic_g, harmonic_alpha):
    theta = np.asarray(theta, dtype=float)
    value = np.zeros_like(theta)
    for n, g, alpha in zip(HARMONIC_N, harmonic_g, harmonic_alpha):
        value += g*np.cos(n*theta + alpha)
    return value


def phase_row_histograms(
        logpolar_map, m_arms, pitch_angle, field="radial_residual",
        n_phase=RIDGE_N_PHASE):
    """Accumulate coverage-weighted values by radial row and spiral phase."""
    m_value = float(m_arms)
    pitch_value = float(pitch_angle)
    n_phase = int(n_phase)
    if not np.isfinite(m_value) or m_value <= 0.0 or m_value != int(m_value):
        raise ValueError("m_arms must be a positive integer")
    if not np.isfinite(pitch_value) or abs(np.sin(np.deg2rad(pitch_value))) < 1.0e-12:
        raise ValueError("pitch_angle must be finite and non-zero")
    if n_phase < 1:
        raise ValueError("n_phase must be positive")
    if field not in logpolar_map:
        raise ValueError(f"Missing log-polar field: {field}")

    u = np.asarray(logpolar_map["u"], dtype=float)
    phi = np.asarray(logpolar_map["phi"], dtype=float)
    values = np.asarray(logpolar_map[field], dtype=float)
    coverage = np.asarray(logpolar_map["coverage"], dtype=float)
    valid = np.asarray(logpolar_map["valid"], dtype=bool)
    expected_shape = (u.size, phi.size)
    if u.ndim != 1 or phi.ndim != 1:
        raise ValueError("Log-polar coordinates must be one-dimensional")
    if values.shape != expected_shape or coverage.shape != expected_shape:
        raise ValueError("Log-polar fields do not match coordinate axes")
    if valid.shape != expected_shape:
        raise ValueError("Log-polar valid mask has the wrong shape")
    if not np.isfinite(u).all() or not np.isfinite(phi).all():
        raise ValueError("Log-polar coordinates must be finite")
    if not np.isfinite(coverage).all() or np.any(coverage < 0.0):
        raise ValueError("Log-polar coverage must be finite and non-negative")

    good = valid & np.isfinite(values) & (coverage > 0.0)
    uu, pp = np.meshgrid(u, phi, indexing="ij")
    row_index = np.broadcast_to(
        np.arange(u.size)[:, None], expected_shape)[good]
    base_phase = np.mod(
        (int(m_value) / np.tan(np.deg2rad(pitch_value))) * uu[good]
        - int(m_value) * pp[good],
        2.0 * np.pi,
    )
    phase_index = (
        np.floor(base_phase / (2.0 * np.pi) * n_phase).astype(int)
        % n_phase
    )
    joint_index = row_index * n_phase + phase_index
    size = u.size * n_phase
    weighted = np.bincount(
        joint_index,
        weights=coverage[good] * values[good],
        minlength=size,
    ).reshape(u.size, n_phase)
    support = np.bincount(
        joint_index,
        weights=coverage[good],
        minlength=size,
    ).reshape(u.size, n_phase)
    return weighted, support


def variable_width_ridge_response(
        weighted_hist, support_hist, u_centres, m_arms, pitch_angle,
        core_width_kpc=RIDGE_CORE_WIDTH_KPC,
        broad_ratio=RIDGE_BROAD_RATIO,
        n_radial_bands=LOGPOLAR_N_RADIAL_BANDS):
    """Measure a narrow-minus-broad ridge response at fixed physical width."""
    weighted_hist = np.asarray(weighted_hist, dtype=float)
    support_hist = np.asarray(support_hist, dtype=float)
    u_centres = np.asarray(u_centres, dtype=float)
    if weighted_hist.ndim != 2 or support_hist.shape != weighted_hist.shape:
        raise ValueError("Weighted and support histograms must share a 2-D shape")
    if not np.isfinite(weighted_hist).all():
        raise ValueError("Weighted histogram must be finite")
    if not np.isfinite(support_hist).all() or np.any(support_hist < 0.0):
        raise ValueError("Support histogram must be finite and non-negative")
    n_u, n_phase = weighted_hist.shape
    if u_centres.ndim != 1 or u_centres.size != n_u:
        raise ValueError("u_centres must match the histogram radial axis")
    if not np.isfinite(u_centres).all():
        raise ValueError("u_centres must be finite")
    n_radial_bands = int(n_radial_bands)
    if n_radial_bands < 1 or n_u % n_radial_bands:
        raise ValueError("Radial rows must divide evenly into positive bands")
    if n_phase < 7:
        raise ValueError("Phase histogram is too short for the width clamps")

    m_value = float(m_arms)
    pitch_value = float(pitch_angle)
    core_width_kpc = float(core_width_kpc)
    broad_ratio = float(broad_ratio)
    if not np.isfinite(m_value) or m_value <= 0.0 or m_value != int(m_value):
        raise ValueError("m_arms must be a positive integer")
    sine_pitch = abs(np.sin(np.deg2rad(pitch_value)))
    if not np.isfinite(pitch_value) or sine_pitch < 1.0e-12:
        raise ValueError("pitch_angle must be finite and non-zero")
    if not np.isfinite(core_width_kpc) or core_width_kpc <= 0.0:
        raise ValueError("core_width_kpc must be finite and positive")
    if not np.isfinite(broad_ratio) or broad_ratio <= 1.0:
        raise ValueError("broad_ratio must be finite and greater than one")
    if not np.isfinite(R_REF_KPC) or R_REF_KPC <= 0.0:
        raise ValueError("R_REF_KPC must be finite and positive")

    rows_per_band = n_u // n_radial_bands
    weighted_band = weighted_hist.reshape(
        n_radial_bands, rows_per_band, n_phase).sum(axis=1)
    support_band = support_hist.reshape(
        n_radial_bands, rows_per_band, n_phase).sum(axis=1)
    u_band = u_centres.reshape(
        n_radial_bands, rows_per_band).mean(axis=1)
    radius_band = R_REF_KPC * np.exp(u_band)
    if not np.isfinite(radius_band).all() or np.any(radius_band <= 0.0):
        raise ValueError("Radial-band radii must be finite and positive")

    sigma_phase = (
        int(m_value) * core_width_kpc / (radius_band * sine_pitch))
    raw_sigma_bins = sigma_phase / (2.0 * np.pi) * n_phase
    low_limit = 0.65
    high_limit = n_phase / 10.0
    low_clamp_count = int(np.count_nonzero(raw_sigma_bins < low_limit))
    high_clamp_count = int(np.count_nonzero(raw_sigma_bins > high_limit))
    sigma_bins = np.clip(raw_sigma_bins, low_limit, high_limit)

    narrow_mean = np.full_like(weighted_band, np.nan, dtype=float)
    broad_mean = np.full_like(weighted_band, np.nan, dtype=float)
    narrow_support = np.zeros_like(support_band, dtype=float)
    for radial_index, sigma_bin in enumerate(sigma_bins):
        narrow_weighted = gaussian_filter1d(
            weighted_band[radial_index], sigma_bin, mode="wrap")
        narrow_denominator = gaussian_filter1d(
            support_band[radial_index], sigma_bin, mode="wrap")
        broad_weighted = gaussian_filter1d(
            weighted_band[radial_index], broad_ratio * sigma_bin, mode="wrap")
        broad_denominator = gaussian_filter1d(
            support_band[radial_index], broad_ratio * sigma_bin, mode="wrap")
        narrow_mean[radial_index] = np.divide(
            narrow_weighted,
            narrow_denominator,
            out=np.full(n_phase, np.nan),
            where=narrow_denominator > 0.05,
        )
        broad_mean[radial_index] = np.divide(
            broad_weighted,
            broad_denominator,
            out=np.full(n_phase, np.nan),
            where=broad_denominator > 0.05,
        )
        narrow_support[radial_index] = narrow_denominator
    return {
        "response": narrow_mean - broad_mean,
        "narrow_mean": narrow_mean,
        "broad_mean": broad_mean,
        "support": narrow_support,
        "sigma_bins": sigma_bins,
        "low_clamp_count": low_clamp_count,
        "high_clamp_count": high_clamp_count,
    }


def aggregate_radial_response(
        response, support, min_radial_fraction=0.35):
    """Combine radial-band responses with one equal vote per valid band."""
    response = np.asarray(response, dtype=float)
    support = np.asarray(support, dtype=float)
    min_radial_fraction = float(min_radial_fraction)
    if response.ndim != 2 or support.shape != response.shape:
        raise ValueError("Response and support must share a 2-D shape")
    if not np.isfinite(support).all() or np.any(support < 0.0):
        raise ValueError("Response support must be finite and non-negative")
    if not np.isfinite(min_radial_fraction) or not 0.0 <= min_radial_fraction <= 1.0:
        raise ValueError("min_radial_fraction must lie in [0, 1]")

    good = np.isfinite(response) & (support > 0.05)
    count = good.sum(axis=0)
    total = np.where(good, response, 0.0).sum(axis=0)
    mean_response = np.divide(
        total,
        count,
        out=np.full(response.shape[1], np.nan),
        where=count > 0,
    )
    positive_count = (good & (response > 0.0)).sum(axis=0)
    positive_fraction = np.divide(
        positive_count,
        count,
        out=np.full(response.shape[1], np.nan),
        where=count > 0,
    )
    median_response = np.array([
        np.median(response[good[:, phase_index], phase_index])
        if count[phase_index] else np.nan
        for phase_index in range(response.shape[1])
    ])
    radial_fraction = count / response.shape[0]
    score = mean_response * positive_fraction + 0.5 * median_response
    score[radial_fraction < min_radial_fraction] = np.nan
    return {
        "score": score,
        "mean": mean_response,
        "median": median_response,
        "positive_fraction": positive_fraction,
        "radial_fraction": radial_fraction,
    }


def ridge_curve_for_map(
        logpolar_map, m_arms, pitch_angle,
        core_width_kpc=RIDGE_CORE_WIDTH_KPC,
        min_radial_fraction=0.35):
    """Return the radial ridge response and its phase-wise aggregate."""
    weighted, support = phase_row_histograms(
        logpolar_map,
        m_arms,
        pitch_angle,
        field="radial_residual",
    )
    response = variable_width_ridge_response(
        weighted,
        support,
        logpolar_map["u"],
        m_arms,
        pitch_angle,
        core_width_kpc=core_width_kpc,
    )
    aggregate = aggregate_radial_response(
        response["response"],
        response["support"],
        min_radial_fraction=min_radial_fraction,
    )
    return response, aggregate


def conditional_ridge_search(
        raw_state, m_compare=M_COMPARE, pitch_grid=RIDGE_PITCH_GRID_DEG,
        core_width_kpc=RIDGE_CORE_WIDTH_KPC,
        guard_bins=RIDGE_GUARD_BINS,
        allow_no_acceptance=False):
    """Return one independently validated negative-winding geometry per m."""
    try:
        m_values = np.asarray(m_compare, dtype=float)
    except (TypeError, ValueError) as exc:
        raise ValueError("m_compare must contain positive integers") from exc
    if m_values.ndim != 1 or m_values.size == 0:
        raise ValueError("m_compare must be a non-empty one-dimensional array")
    if (not np.isfinite(m_values).all()
            or np.any(m_values <= 0.0)
            or not np.equal(m_values, np.floor(m_values)).all()):
        raise ValueError("m_compare must contain positive integers")
    m_values = m_values.astype(int)
    if np.unique(m_values).size != m_values.size:
        raise ValueError("m_compare must not contain duplicates")

    try:
        pitch_values = np.asarray(pitch_grid, dtype=float)
    except (TypeError, ValueError) as exc:
        raise ValueError("pitch_grid must contain finite negative values") from exc
    if pitch_values.ndim != 1 or pitch_values.size == 0:
        raise ValueError("pitch_grid must be a non-empty one-dimensional array")
    if not np.isfinite(pitch_values).all() or np.any(pitch_values >= 0.0):
        raise ValueError("pitch_grid must contain finite negative values")
    if np.unique(pitch_values).size != pitch_values.size:
        raise ValueError("pitch_grid must not contain duplicates")

    core_width_kpc = float(core_width_kpc)
    guard_value = float(guard_bins)
    if not np.isfinite(core_width_kpc) or core_width_kpc <= 0.0:
        raise ValueError("core_width_kpc must be finite and positive")
    if (not np.isfinite(guard_value)
            or guard_value < 0.0
            or guard_value != int(guard_value)):
        raise ValueError("guard_bins must be a non-negative integer")
    guard_bins = int(guard_value)

    full_map = preprocess_logpolar_raw(
        raw_state, allowed_phi=None, include_local=False)
    phase_values = np.linspace(
        0.0, 2.0 * np.pi, RIDGE_N_PHASE, endpoint=False)
    branch_normalization = "mean_not_sum"
    pitch_min = float(np.min(pitch_values))
    pitch_max = float(np.max(pitch_values))
    ridge_fold_columns = [
        "candidate_index", "m_arms", "pitch_angle",
        "held_sector", "train_phase_index", "train_Theta0",
        "train_score", "test_score", "test_radial_fraction",
        "valid_test",
    ]
    records = []
    for m_arms in m_values:
        for pitch_angle in pitch_values:
            response, aggregate = ridge_curve_for_map(
                full_map,
                int(m_arms),
                float(pitch_angle),
                core_width_kpc=core_width_kpc,
            )
            if not np.isfinite(aggregate["score"]).any():
                continue
            phase_index = int(np.nanargmax(aggregate["score"]))
            finite_narrow = response["narrow_mean"][:, phase_index]
            finite_narrow = finite_narrow[np.isfinite(finite_narrow)]
            finite_broad = response["broad_mean"][:, phase_index]
            finite_broad = finite_broad[np.isfinite(finite_broad)]
            records.append({
                "m_arms": int(m_arms),
                "pitch_angle": float(pitch_angle),
                "winding_sign": -1,
                "Theta0": float(
                    (-phase_values[phase_index]) % (2.0 * np.pi)),
                "full_phase_index": phase_index,
                "core_width_kpc": core_width_kpc,
                "full_ridge_score": float(
                    aggregate["score"][phase_index]),
                "narrow_mean": (
                    float(np.mean(finite_narrow))
                    if finite_narrow.size else np.nan),
                "broad_flank_mean": (
                    float(np.mean(finite_broad))
                    if finite_broad.size else np.nan),
                "mean_radial_response": float(
                    aggregate["mean"][phase_index]),
                "median_radial_response": float(
                    aggregate["median"][phase_index]),
                "positive_radial_fraction": float(
                    aggregate["positive_fraction"][phase_index]),
                "radial_coverage": float(
                    aggregate["radial_fraction"][phase_index]),
                "low_width_clamp_count": int(
                    response["low_clamp_count"]),
                "high_width_clamp_count": int(
                    response["high_clamp_count"]),
                "pitch_boundary": bool(
                    np.isclose(pitch_angle, pitch_min)
                    or np.isclose(pitch_angle, pitch_max)),
                "branch_normalization": branch_normalization,
                "held_out_score": np.nan,
                "held_out_score_std": np.nan,
                "held_out_coverage_fraction": 0.0,
                "phase_stability": np.nan,
                "valid_held_out": 0,
                "validated_score": np.nan,
            })

    histogram_cache_payload_bytes = 0
    candidate_table = pd.DataFrame(records)
    if candidate_table.empty:
        if not allow_no_acceptance:
            raise RuntimeError(
                "No ridge candidate had finite full-map support")
        empty_geometry_rows = []
        for m_arms in m_values:
            empty_geometry_rows.append({
                "m_arms": int(m_arms),
                "pitch_angle": float(pitch_values[0]),
                "winding_sign": -1,
                "Theta0": np.nan,
                "full_phase_index": -1,
                "core_width_kpc": core_width_kpc,
                "full_ridge_score": np.nan,
                "held_out_score": np.nan,
                "held_out_score_std": np.nan,
                "held_out_coverage_fraction": 0.0,
                "phase_stability": np.nan,
                "valid_held_out": 0,
                "validated_score": np.nan,
                "pitch_boundary": True,
                "accepted": False,
                "acceptance_reason": (
                    "below_threshold: no_finite_full_map_candidate; "
                    f"valid_held_out=0/{RIDGE_N_SECTORS}; "
                    f"required>={RIDGE_MIN_HELD_OUT_SECTORS}; "
                    "validated_score=nan"
                ),
                "geometry_source": (
                    "leakage_controlled_negative_winding_ridge"),
            })
        return (
            pd.DataFrame(empty_geometry_rows),
            candidate_table,
            pd.DataFrame(columns=ridge_fold_columns),
            histogram_cache_payload_bytes,
        )

    shortlist_indices = []
    for m_arms in m_values:
        family = candidate_table.loc[
            candidate_table["m_arms"] == int(m_arms)]
        shortlist_indices.extend(
            family.nlargest(
                RIDGE_SHORTLIST_PER_FAMILY,
                "full_ridge_score",
            ).index.tolist()
        )

    folds = build_sector_fold_maps(
        raw_state,
        n_sectors=RIDGE_N_SECTORS,
        guard_bins=guard_bins,
    )
    fold_records = []
    for record_index in shortlist_indices:
        m_arms = int(candidate_table.at[record_index, "m_arms"])
        pitch_angle = float(
            candidate_table.at[record_index, "pitch_angle"])
        held_scores = []
        train_phases = []
        for fold in folds:
            train_response, train_aggregate = ridge_curve_for_map(
                fold["train"],
                m_arms,
                pitch_angle,
                core_width_kpc=core_width_kpc,
            )
            train_valid = np.isfinite(train_aggregate["score"])
            train_phase_index = -1
            train_theta0 = np.nan
            train_score = np.nan
            test_score = np.nan
            test_radial_fraction = np.nan
            fold_valid = False
            if np.any(train_valid):
                train_phase_index = int(
                    np.nanargmax(train_aggregate["score"]))
                train_theta0 = float(
                    (-phase_values[train_phase_index])
                    % (2.0 * np.pi))
                train_score = float(
                    train_aggregate["score"][train_phase_index])
                _, test_aggregate = ridge_curve_for_map(
                    fold["test"],
                    m_arms,
                    pitch_angle,
                    core_width_kpc=core_width_kpc,
                    min_radial_fraction=0.08,
                )
                if np.isfinite(
                        test_aggregate["score"][train_phase_index]):
                    test_score = float(
                        test_aggregate["score"][train_phase_index])
                    test_radial_fraction = float(
                        test_aggregate["radial_fraction"][
                            train_phase_index])
                    fold_valid = True
                    held_scores.append(test_score)
                    train_phases.append(train_theta0)
            fold_records.append({
                "candidate_index": int(record_index),
                "m_arms": m_arms,
                "pitch_angle": pitch_angle,
                "held_sector": int(fold["held_sector"]),
                "train_phase_index": train_phase_index,
                "train_Theta0": train_theta0,
                "train_score": train_score,
                "test_score": test_score,
                "test_radial_fraction": test_radial_fraction,
                "valid_test": fold_valid,
            })

        valid_held_out = len(held_scores)
        if valid_held_out:
            phase_stability = float(np.abs(np.mean(
                np.exp(1j * np.asarray(train_phases)))))
            held_out_score = float(np.median(held_scores))
            coverage_fraction = valid_held_out / len(folds)
            candidate_table.at[
                record_index, "held_out_score"] = held_out_score
            candidate_table.at[
                record_index, "held_out_score_std"] = float(
                    np.std(held_scores))
            candidate_table.at[
                record_index, "held_out_coverage_fraction"] = (
                    coverage_fraction)
            candidate_table.at[
                record_index, "phase_stability"] = phase_stability
            candidate_table.at[
                record_index, "valid_held_out"] = valid_held_out
            candidate_table.at[
                record_index, "validated_score"] = (
                    max(held_out_score, 0.0)
                    * coverage_fraction
                    * phase_stability
                )

    geometry_rows = []
    missing_modes = []
    for m_arms in m_values:
        family_all = candidate_table.loc[
            candidate_table["m_arms"] == int(m_arms)
        ].copy()
        passing = family_all.loc[
            (family_all["valid_held_out"]
               >= RIDGE_MIN_HELD_OUT_SECTORS)
            & (family_all["validated_score"] > 0.0)
        ].copy()
        if not passing.empty:
            best = passing.sort_values(
                ["validated_score", "full_ridge_score"],
                ascending=False,
            ).iloc[0].copy()
            best["accepted"] = True
            best["acceptance_reason"] = "passed"
        else:
            missing_modes.append(int(m_arms))
            if not allow_no_acceptance:
                continue
            held_evaluated = family_all.loc[
                (family_all["valid_held_out"] > 0)
                & np.isfinite(family_all["held_out_score"])
            ].copy()
            if not held_evaluated.empty:
                best = held_evaluated.sort_values(
                    ["validated_score", "full_ridge_score"],
                    ascending=False,
                    na_position="last",
                ).iloc[0].copy()
            elif not family_all.empty:
                best = family_all.sort_values(
                    "full_ridge_score", ascending=False
                ).iloc[0].copy()
            else:
                best = pd.Series({
                    "m_arms": int(m_arms),
                    "pitch_angle": float(pitch_values[0]),
                    "winding_sign": -1,
                    "Theta0": np.nan,
                    "full_ridge_score": np.nan,
                    "held_out_score": np.nan,
                    "phase_stability": np.nan,
                    "valid_held_out": 0,
                    "validated_score": np.nan,
                })
            held_count = int(best.get("valid_held_out", 0))
            validated = float(best.get("validated_score", np.nan))
            validated_text = (
                f"{validated:.6g}" if np.isfinite(validated) else "nan")
            best["accepted"] = False
            best["acceptance_reason"] = (
                "below_threshold: "
                f"valid_held_out={held_count}/"
                f"{RIDGE_N_SECTORS}; "
                f"required>={RIDGE_MIN_HELD_OUT_SECTORS}; "
                f"validated_score={validated_text}"
            )
        best["geometry_source"] = (
            "leakage_controlled_negative_winding_ridge")
        geometry_rows.append(best)

    if missing_modes and not allow_no_acceptance:
        raise RuntimeError(
            "No accepted conditional ridge geometry for m="
            + ",".join(str(value) for value in missing_modes)
        )

    ridge_geometry_table = pd.DataFrame(geometry_rows)
    if not ridge_geometry_table.empty:
        ridge_geometry_table = (
            ridge_geometry_table.sort_values("m_arms")
            .reset_index(drop=True)
        )
    ranked_candidates = (
        candidate_table.sort_values(
            ["m_arms", "validated_score", "full_ridge_score"],
            ascending=[True, False, False],
            na_position="last",
        ).reset_index(drop=True)
    )
    ridge_fold_table = pd.DataFrame(
        fold_records, columns=ridge_fold_columns)
    return (
        ridge_geometry_table,
        ranked_candidates,
        ridge_fold_table,
        histogram_cache_payload_bytes,
    )

## 9. Fit KTZ-compatible source profiles at frozen ridge geometries

KTZ here means an axisymmetric exponential background multiplied by a periodic source-rate modulation. The fitted `h_R` is the radial e-folding length of that background, `eta` is the modulation strength, and the fixed first three harmonics describe the periodic profile shape. Arm number, pitch, and `Theta0` are supplied by the leakage-controlled ridge stage and are never optimizer variables. Each of the three ridge rows receives its own source-profile fit; the m=2 fit is retained as a rejected conditional sensitivity case, not relabelled as a detection.

SciPy's integer `active_mask` is retained with named lower/upper bound directions. If it places `g2` or `g3` on a bound, `harmonic_bound_limited` is true: those harmonic coefficients are constraint-limited boundary results rather than well-determined measurements. The approved bounds are not loosened in response.

In [ ]:
def fit_ktz_profile_fixed_geometry(table, geometry, radial_guess):
    """Fit only the KTZ source profile at one supplied ridge geometry."""
    required_columns = (
        "radius_kpc", "azimuth_rad", "sfr_linear", "area_pix")
    missing_columns = [
        column for column in required_columns if column not in table]
    if missing_columns:
        raise ValueError(f"Fit table is missing columns: {missing_columns}")
    radius = table["radius_kpc"].to_numpy(float)
    azimuth = table["azimuth_rad"].to_numpy(float)
    sfr_linear = table["sfr_linear"].to_numpy(float)
    area_pix = table["area_pix"].to_numpy(float)
    if radius.ndim != 1 or radius.size == 0:
        raise ValueError("Fit table must contain one-dimensional data")
    if not (azimuth.shape == sfr_linear.shape == area_pix.shape
            == radius.shape):
        raise ValueError("Fit-table columns must have matching shapes")
    if not np.isfinite(radius).all() or np.any(radius <= 0.0):
        raise ValueError("radius_kpc must be finite and positive")
    if not np.isfinite(azimuth).all():
        raise ValueError("azimuth_rad must be finite")
    if not np.isfinite(sfr_linear).all() or np.any(sfr_linear <= 0.0):
        raise ValueError("sfr_linear must be finite and positive")
    if not np.isfinite(area_pix).all() or np.any(area_pix <= 0.0):
        raise ValueError("area_pix must be finite and positive")

    required_geometry = (
        "m_arms", "pitch_angle", "Theta0", "geometry_source",
        "accepted", "acceptance_reason")
    missing_geometry = [
        key for key in required_geometry if key not in geometry]
    if missing_geometry:
        raise ValueError(
            f"Fixed geometry is missing fields: {missing_geometry}")
    m_value = float(geometry["m_arms"])
    pitch_value = float(geometry["pitch_angle"])
    theta0_value = float(geometry["Theta0"])
    if (not np.isfinite(m_value) or m_value <= 0.0
            or m_value != int(m_value)):
        raise ValueError("m_arms must be a finite positive integer")
    if (not np.isfinite(pitch_value)
            or abs(np.sin(np.deg2rad(pitch_value))) < 1.0e-12):
        raise ValueError("pitch_angle must be finite and non-zero")
    if not np.isfinite(theta0_value):
        raise ValueError("Theta0 must be finite")
    if (not isinstance(geometry["geometry_source"], str)
            or not geometry["geometry_source"]):
        raise ValueError("geometry_source must be a non-empty string")
    if not isinstance(geometry["accepted"], (bool, np.bool_)):
        raise ValueError("accepted must be boolean")
    if not isinstance(geometry["acceptance_reason"], str):
        raise ValueError("acceptance_reason must be a string")

    required_radial = ("lambda0_0", "h_R")
    missing_radial = [
        key for key in required_radial if key not in radial_guess]
    if missing_radial:
        raise ValueError(
            f"Radial guess is missing fields: {missing_radial}")
    lambda_guess = float(radial_guess["lambda0_0"])
    h_guess = float(radial_guess["h_R"])
    if not np.isfinite(lambda_guess) or lambda_guess <= 0.0:
        raise ValueError("Radial lambda0_0 guess must be positive")
    if not np.isfinite(h_guess) or h_guess <= 0.0:
        raise ValueError("Radial h_R guess must be positive")
    if not np.array_equal(
            np.asarray(HARMONIC_N), np.array([1, 2, 3])):
        raise ValueError("HARMONIC_N must be exactly [1, 2, 3]")

    observed_log_sfr = np.log(sfr_linear)
    weights = np.sqrt(area_pix)
    median_weight = float(np.median(weights))
    if not np.isfinite(median_weight) or median_weight <= 0.0:
        raise ValueError("Median square-root area weight is invalid")
    weights = weights / median_weight
    theta = spiral_phase(
        radius, azimuth, int(m_value), pitch_value, theta0_value)
    if not np.isfinite(theta).all():
        raise ValueError("Fixed-geometry phase is not finite")
    dense_theta = np.linspace(
        -np.pi, np.pi, 8192, endpoint=False)

    initial_log_lambda = float(np.clip(
        np.log(lambda_guess), -50.0 + 1.0e-10, 50.0 - 1.0e-10))
    initial_log_h = float(np.clip(
        np.log(h_guess),
        np.log(0.2) + 1.0e-10, np.log(30.0) - 1.0e-10))
    optimizer_initial = np.array([
        initial_log_lambda, initial_log_h, np.log(0.2),
        0.3, 0.15, 0.0, 0.0,
    ])
    lower_bounds = np.array([
        -50.0, np.log(0.2), np.log(1.0e-4),
        0.0, 0.0, -np.pi, -np.pi,
    ])
    upper_bounds = np.array([
        50.0, np.log(30.0), np.log(2.0),
        1.5, 1.5, np.pi, np.pi,
    ])

    def unpack_profile(parameters):
        log_lambda0, log_h, log_eta, g2, g3, alpha2, alpha3 = parameters
        harmonic_g = np.array([1.0, g2, g3])
        harmonic_alpha = np.array([0.0, alpha2, alpha3])
        return (
            float(np.exp(log_lambda0)),
            float(np.exp(log_h)),
            float(np.exp(log_eta)),
            harmonic_g,
            harmonic_alpha,
        )

    def residual(parameters):
        lambda0_0, h_r, eta, harmonic_g, harmonic_alpha = (
            unpack_profile(parameters))
        rate_factor = 1.0 + eta * arm_profile(
            theta, harmonic_g, harmonic_alpha)
        safe_rate_factor = np.clip(rate_factor, 1.0e-12, None)
        model_log_sfr = (
            np.log(lambda0_0) - radius / h_r
            + np.log(safe_rate_factor))
        weighted_log_residual = weights * (
            observed_log_sfr - model_log_sfr)
        dense_rate_factor = 1.0 + eta * arm_profile(
            dense_theta, harmonic_g, harmonic_alpha)
        positivity_penalty = 1.0e4 * np.clip(
            1.0e-3 - dense_rate_factor, 0.0, None)
        return np.concatenate([
            weighted_log_residual, positivity_penalty])

    result = least_squares(
        residual,
        optimizer_initial,
        bounds=(lower_bounds, upper_bounds),
        loss="soft_l1",
        f_scale=0.25,
        max_nfev=3000,
    )
    if not result.success:
        raise RuntimeError(f"Fixed-geometry KTZ fit failed: {result.message}")
    lambda0_0, h_r, eta, harmonic_g, harmonic_alpha = (
        unpack_profile(result.x))
    dense_rate_factor = 1.0 + eta * arm_profile(
        dense_theta, harmonic_g, harmonic_alpha)
    if not np.isfinite(dense_rate_factor).all():
        raise RuntimeError("Fitted dense KTZ rate factor is not finite")
    minimum_rate_factor = float(np.min(dense_rate_factor))
    if minimum_rate_factor <= 0.0:
        raise RuntimeError("Fitted dense KTZ rate factor is not positive")
    fitted_rate_factor = 1.0 + eta * arm_profile(
        theta, harmonic_g, harmonic_alpha)
    fitted_log_sfr = (
        np.log(lambda0_0) - radius / h_r
        + np.log(np.clip(fitted_rate_factor, 1.0e-12, None)))
    weighted_log_residual = weights * (
        observed_log_sfr - fitted_log_sfr)
    parameter_names = [
        "log_lambda0_0", "log_h_R", "log_eta",
        "g2", "g3", "alpha2", "alpha3",
    ]
    active_mask_array = np.asarray(result.active_mask, dtype=int)
    if (active_mask_array.shape != (len(parameter_names),)
            or not np.isin(active_mask_array, [-1, 0, 1]).all()):
        raise RuntimeError(
            "Optimizer returned an invalid active-bound mask")
    active_mask = [int(value) for value in active_mask_array]
    active_bound_parameters = {
        name: ("lower" if direction < 0 else "upper")
        for name, direction in zip(parameter_names, active_mask)
        if direction != 0
    }
    harmonic_bound_limited = bool(
        active_mask[3] != 0 or active_mask[4] != 0)
    njev_value = getattr(result, "njev", None)
    return {
        "lambda0_0": lambda0_0,
        "h_R": h_r,
        "eta": eta,
        "harmonic_n": np.asarray(HARMONIC_N, dtype=int).copy(),
        "harmonic_g": harmonic_g,
        "harmonic_alpha": harmonic_alpha,
        "m_arms": int(m_value),
        "pitch_angle": pitch_value,
        "Theta0": theta0_value,
        "geometry_source": geometry["geometry_source"],
        "ridge_accepted": bool(geometry["accepted"]),
        "ridge_acceptance_reason": geometry["acceptance_reason"],
        "weighted_log_sse": float(np.sum(weighted_log_residual**2)),
        "robust_cost": float(2.0 * result.cost),
        "minimum_rate_factor": minimum_rate_factor,
        "parameter_names": parameter_names,
        "active_mask": active_mask,
        "active_bound_parameters": active_bound_parameters,
        "harmonic_bound_limited": harmonic_bound_limited,
        "optimality": float(result.optimality),
        "nfev": int(result.nfev),
        "njev": (None if njev_value is None else int(njev_value)),
        "optimizer_status": int(result.status),
        "optimizer_message": str(result.message),
        "success": bool(result.success),
        "message": str(result.message),
    }


def enumerate_profile_maxima(model, n_grid=65536):
    """Enumerate refined circular maxima of a positive KTZ profile."""
    n_grid_value = float(n_grid)
    if (not np.isfinite(n_grid_value) or n_grid_value < 32
            or n_grid_value != int(n_grid_value)):
        raise ValueError("n_grid must be an integer of at least 32")
    n_grid = int(n_grid_value)
    required = ("eta", "harmonic_n", "harmonic_g", "harmonic_alpha")
    missing = [key for key in required if key not in model]
    if missing:
        raise ValueError(f"Profile model is missing fields: {missing}")
    eta = float(model["eta"])
    harmonic_n = np.asarray(model["harmonic_n"], dtype=float)
    harmonic_g = np.asarray(model["harmonic_g"], dtype=float)
    harmonic_alpha = np.asarray(
        model["harmonic_alpha"], dtype=float)
    if not np.isfinite(eta) or eta <= 0.0:
        raise ValueError("eta must be finite and positive")
    if (harmonic_n.ndim != 1 or harmonic_n.size == 0
            or harmonic_g.shape != harmonic_n.shape
            or harmonic_alpha.shape != harmonic_n.shape):
        raise ValueError("Harmonic arrays must be nonempty matching vectors")
    if (not np.isfinite(harmonic_n).all()
            or not np.isfinite(harmonic_g).all()
            or not np.isfinite(harmonic_alpha).all()):
        raise ValueError("Harmonic arrays must be finite")
    if (np.any(harmonic_n <= 0.0)
            or not np.equal(harmonic_n, np.floor(harmonic_n)).all()):
        raise ValueError("Harmonic orders must be positive integers")

    def evaluate(theta_values):
        theta_values = np.asarray(theta_values, dtype=float)
        h_profile = np.sum(
            harmonic_g[:, None] * np.cos(
                harmonic_n[:, None] * theta_values[None, :]
                + harmonic_alpha[:, None]),
            axis=0,
        )
        return h_profile, 1.0 + eta * h_profile

    theta_grid = np.linspace(
        -np.pi, np.pi, n_grid, endpoint=False)
    h_grid, rate_grid = evaluate(theta_grid)
    if (not np.isfinite(h_grid).all()
            or not np.isfinite(rate_grid).all()):
        raise ValueError("Periodic profile is not finite")
    if np.min(rate_grid) <= 0.0:
        raise ValueError("Periodic profile must remain strictly positive")
    local_maximum = (
        (rate_grid >= np.roll(rate_grid, 1))
        & (rate_grid > np.roll(rate_grid, -1)))
    peak_indices = np.flatnonzero(local_maximum)
    if peak_indices.size == 0:
        raise RuntimeError("Periodic profile has no local maximum")
    step = 2.0 * np.pi / n_grid
    rows = []
    for peak_index in peak_indices:
        centre = float(theta_grid[peak_index])

        def negative_rate(theta_value):
            h_value = float(np.sum(
                harmonic_g * np.cos(
                    harmonic_n * theta_value + harmonic_alpha)))
            return -(1.0 + eta * h_value)

        refined = minimize_scalar(
            negative_rate,
            bounds=(centre - step, centre + step),
            method="bounded",
            options={"xatol": 1.0e-13},
        )
        if not refined.success:
            raise RuntimeError(
                f"Profile-maximum refinement failed: {refined.message}")
        theta_peak = float(wrap_angle(refined.x))
        h_peak = float(np.sum(
            harmonic_g * np.cos(
                harmonic_n * theta_peak + harmonic_alpha)))
        rate_factor = float(1.0 + eta * h_peak)
        rows.append({
            "theta_peak": theta_peak,
            "h_peak": h_peak,
            "rate_factor": rate_factor,
            "enhanced_above_background": bool(rate_factor > 1.0),
        })
    maxima = pd.DataFrame(rows)
    maxima = maxima.sort_values(
        "rate_factor", ascending=False).reset_index(drop=True)
    if maxima.empty:
        raise RuntimeError("Periodic profile maximum table is empty")
    return maxima

## 10. Synthetic ridge-recovery for arm number, winding, and phase

These masked synthetic SFR fields combine a known exponential radial decline with narrow logarithmic ridges. Three independent negative-winding cases cover the requested two-, three-, and four-arm families. The production ridge selector must recover the fixed arm number, pitch within two degrees, and phase within five degrees. A separate scaling check verifies that multiplying both the weighted histogram and its support leaves the mean-normalized ridge response unchanged.

In [ ]:
def synthetic_sfr_pixels(m_arms, pitch_angle, theta0, seed,
                         n_pixels=80000, mask_fraction=0.25,
                         arm_amplitude_dex=0.48,
                         ridge_sigma_kpc=0.22, radial_h_kpc=3.5):
    """Masked SFR pixels with a known radial decline and logarithmic ridges."""
    local_rng = np.random.default_rng(seed)
    u = local_rng.uniform(np.log(0.35), np.log(11.0), n_pixels)
    azimuth = local_rng.uniform(-np.pi, np.pi, n_pixels)
    radius = R_REF_KPC * np.exp(u)
    theta = ((m_arms / np.tan(np.deg2rad(pitch_angle))) * u
             - m_arms * azimuth + theta0)
    wrapped = wrap_angle(theta)
    perpendicular_kpc = (np.abs(wrapped) * radius
                         * abs(np.sin(np.deg2rad(pitch_angle))) / m_arms)
    ridge_dex = arm_amplitude_dex * np.exp(
        -0.5 * (perpendicular_kpc / ridge_sigma_kpc)**2)
    lambda0_0 = 0.05
    background_log = (np.log10(lambda0_0)
                      - radius / (np.log(10.0) * radial_h_kpc))
    observed_log = background_log + ridge_dex
    observed_log += local_rng.normal(0.0, 0.035, size=n_pixels)
    keep = local_rng.random(n_pixels) >= mask_fraction
    keep &= ~((azimuth > 0.35) & (azimuth < 0.70) & (radius > 5.5))
    keep &= ~((azimuth > -2.20) & (azimuth < -1.95) & (radius < 2.2))
    pixels = pd.DataFrame({"radius_kpc": radius[keep],
                           "azimuth_rad": azimuth[keep],
                           "log_sfr": observed_log[keep]})
    return pixels, {"lambda0_0": lambda0_0, "h_R": radial_h_kpc}


def synthetic_logpolar(m_arms, pitch_angle, theta0, seed, **kwargs):
    pixels, radial_parameters = synthetic_sfr_pixels(
        m_arms, pitch_angle, theta0, seed, **kwargs)
    return build_log_polar_contrast(pixels, radial_parameters)


synthetic_cases = [(2, -22.0), (3, -30.0), (4, -38.0)]
theta0_true = 0.7
synthetic_recoveries = []
for case_index, (m_true, pitch_true) in enumerate(synthetic_cases):
    synthetic_raw, synthetic_processed = synthetic_logpolar(
        m_true, pitch_true, theta0_true, RNG_SEED + case_index)
    synthetic_geometry, synthetic_candidates, synthetic_folds, (
        synthetic_cache_bytes) = conditional_ridge_search(
            synthetic_raw, m_compare=np.array([m_true], dtype=int),
            pitch_grid=RIDGE_PITCH_GRID_DEG)
    assert len(synthetic_geometry) == 1, synthetic_geometry
    recovered = synthetic_geometry.iloc[0].to_dict()
    pitch_error_deg = abs(recovered["pitch_angle"] - pitch_true)
    phase_error_deg = np.rad2deg(abs(wrap_angle(
        recovered["Theta0"] - theta0_true)))
    synthetic_recoveries.append({
        "m_true": m_true,
        "pitch_true_deg": pitch_true,
        "m_recovered": int(recovered["m_arms"]),
        "pitch_recovered_deg": recovered["pitch_angle"],
        "pitch_error_deg": pitch_error_deg,
        "Theta0_recovered_rad": recovered["Theta0"],
        "phase_error_deg": phase_error_deg,
        "validated_score": recovered["validated_score"],
        "histogram_cache_payload_bytes": synthetic_cache_bytes,
    })
    assert recovered["m_arms"] == m_true, (m_true, recovered)
    assert recovered["pitch_angle"] < 0.0, recovered
    assert pitch_error_deg <= 2.0, recovered
    assert phase_error_deg <= 5.0, (m_true, phase_error_deg, recovered)
    assert synthetic_cache_bytes == 0
synthetic_recovery_table = pd.DataFrame(synthetic_recoveries)
assert len(synthetic_recovery_table) == len(synthetic_cases) == 3
print(synthetic_recovery_table.to_string(index=False))
print("RIDGE_M234_SYNTHETIC_PASS")

test_weighted, test_support = phase_row_histograms(
    synthetic_processed, 4, -38.0)
response_1 = variable_width_ridge_response(
    test_weighted, test_support, synthetic_processed["u"], 4, -38.0
)["response"]
response_6 = variable_width_ridge_response(
    6.0 * test_weighted, 6.0 * test_support,
    synthetic_processed["u"], 4, -38.0)["response"]
assert np.allclose(response_1, response_6, equal_nan=True, atol=1.0e-12)
print("RIDGE_BRANCH_NORMALIZATION_PASS")

## 11. Run the three real conditional geometries and descriptive diagnostics

The real NGC4254 analysis screens the requested m=2, 3, and 4 negative-winding families independently. A family is accepted only when at least ten held sectors are valid and its validated score is positive. If a family misses that fixed rule, its best finite conditional geometry remains visible with accepted=False and a quantitative reason; it is not called a detection, and the threshold is never weakened.

Four independent random-number blocks use seeds 4254, 5254, 6254, and 7254, with eight sequential draws per block. Each draw circularly shifts the unsmoothed weighted sum and coverage together within every radial row before rebuilding leakage-controlled folds. The pooled and blockwise null summaries are descriptive diagnostics only: their null_z values neither select an arm family nor filter the three real geometry rows.

Five fixed physical ridge widths are evaluated with the same conditional acceptance rule. The resulting fifteen rows expose width sensitivity for every requested arm family, including below-threshold cases.

In [ ]:
def scramble_logpolar_raw(raw_state, rng):
    """Circularly shift each raw radial row while preserving weight/coverage pairing."""
    scrambled = {
        key: (np.array(value, copy=True)
              if isinstance(value, np.ndarray) else value)
        for key, value in raw_state.items()
    }
    required = ("raw_weighted_sum", "raw_coverage")
    missing = [key for key in required if key not in scrambled]
    if missing:
        raise ValueError(f"Raw log-polar state is missing keys: {missing}")
    weighted = scrambled["raw_weighted_sum"]
    coverage = scrambled["raw_coverage"]
    if weighted.ndim != 2 or coverage.shape != weighted.shape:
        raise ValueError("Raw weighted sum and coverage must share a 2-D shape")
    n_phi = weighted.shape[1]
    if n_phi < 1:
        raise ValueError("Raw log-polar azimuth axis must be non-empty")
    for row_index in range(weighted.shape[0]):
        shift = int(rng.integers(0, n_phi))
        weighted[row_index] = np.roll(weighted[row_index], shift)
        coverage[row_index] = np.roll(coverage[row_index], shift)
    return scrambled


def build_blocked_null_diagnostics(raw_state):
    """Build four independent blocks of descriptive raw-row null draws."""
    rows = []
    null_role = "descriptive_not_model_selection"
    for block_index, block_seed in enumerate(RIDGE_NULL_BLOCK_SEEDS):
        block_started = perf_counter()
        block_rng = np.random.default_rng(int(block_seed))
        for draw in range(RIDGE_NULL_DRAWS_PER_BLOCK):
            scrambled = scramble_logpolar_raw(raw_state, block_rng)
            geometry, _, _, cache_bytes = conditional_ridge_search(
                scrambled, allow_no_acceptance=True)
            if cache_bytes != 0:
                raise RuntimeError("Blocked null search created a histogram cache")
            if geometry["m_arms"].tolist() != M_COMPARE.tolist():
                raise RuntimeError(
                    "Blocked null search did not return one row per requested m")
            for _, conditional in geometry.iterrows():
                accepted = bool(conditional["accepted"])
                validated_score = float(conditional["validated_score"])
                if accepted and not (
                        np.isfinite(validated_score)
                        and validated_score > 0.0):
                    raise RuntimeError(
                        "Accepted null geometry lacks a positive finite score")
                rows.append({
                    "block_index": int(block_index),
                    "block_seed": int(block_seed),
                    "draw": int(draw),
                    "m_arms": int(conditional["m_arms"]),
                    "accepted": accepted,
                    "acceptance_reason": conditional["acceptance_reason"],
                    "valid_held_out": int(
                        conditional["valid_held_out"]),
                    "validated_score": validated_score,
                    "best_null_score": (
                        validated_score if accepted else 0.0),
                    "null_role": null_role,
                })
        print(
            "RIDGE_NULL_BLOCK_PROGRESS "
            f"block={block_index + 1}/{len(RIDGE_NULL_BLOCK_SEEDS)} "
            f"seed={int(block_seed)} "
            f"elapsed={perf_counter() - block_started:.2f}s"
        )

    ridge_null_draws = pd.DataFrame(rows)
    if len(ridge_null_draws) != 96:
        raise RuntimeError("Blocked null diagnostics must contain 96 rows")
    if not (
            ridge_null_draws.groupby("m_arms").size() == 32).all():
        raise RuntimeError("Each arm family must have 32 blocked null draws")
    if not (
            ridge_null_draws.groupby(
                ["block_index", "block_seed"]).size() == 24).all():
        raise RuntimeError("Each RNG block must contain 24 arm-family rows")
    if not (
            ridge_null_draws.groupby(
                ["block_index", "block_seed", "m_arms"]).size() == 8).all():
        raise RuntimeError("Each block and arm family must contain eight draws")

    pooled_null_summary = (
        ridge_null_draws.groupby("m_arms")["best_null_score"]
        .agg(null_mean="mean", null_std="std", null_count="count")
        .reset_index()
    )
    pooled_null_summary["null_std_floor"] = np.maximum(
        pooled_null_summary["null_std"], 1.0e-6)
    pooled_null_summary["null_role"] = null_role
    null_block_summary = (
        ridge_null_draws.groupby(
            ["block_index", "block_seed", "m_arms"])["best_null_score"]
        .agg(null_mean="mean", null_std="std", null_count="count")
        .reset_index()
    )
    null_block_summary["null_role"] = null_role
    if len(null_block_summary) != 12:
        raise RuntimeError("Blocked null summary must contain 12 rows")
    if not (pooled_null_summary["null_count"] == 32).all():
        raise RuntimeError("Pooled null counts must equal 32 per arm family")
    if not (null_block_summary["null_count"] == 8).all():
        raise RuntimeError("Block null counts must equal eight")
    return ridge_null_draws, pooled_null_summary, null_block_summary


ridge_geometry_table, ridge_candidate_table, ridge_fold_table, (
    ridge_histogram_cache_payload_bytes) = conditional_ridge_search(
    logpolar_raw, allow_no_acceptance=True)
assert ridge_histogram_cache_payload_bytes == 0
assert len(ridge_geometry_table) == len(M_COMPARE) == 3
assert ridge_geometry_table["m_arms"].tolist() == M_COMPARE.tolist()
assert (ridge_geometry_table["pitch_angle"] < 0.0).all()
accepted_geometry = ridge_geometry_table["accepted"].astype(bool)
assert (
    ridge_geometry_table.loc[accepted_geometry, "valid_held_out"]
    >= RIDGE_MIN_HELD_OUT_SECTORS
).all()
assert (
    ridge_geometry_table.loc[accepted_geometry, "validated_score"] > 0.0
).all()
assert (
    ridge_geometry_table.loc[
        ~accepted_geometry, "acceptance_reason"]
    .str.startswith("below_threshold:")
).all()
m2_evidence = ridge_geometry_table.loc[
    ridge_geometry_table["m_arms"] == 2].iloc[0]
print(
    "m=2 conditional evidence: "
    f"pitch={m2_evidence['pitch_angle']:.1f} deg, "
    f"held={int(m2_evidence['valid_held_out'])}/"
    f"{RIDGE_N_SECTORS}, "
    f"score={m2_evidence['validated_score']:.6g}, "
    f"accepted={bool(m2_evidence['accepted'])}, "
    f"reason={m2_evidence['acceptance_reason']}"
)
display(ridge_geometry_table[[
    "m_arms", "pitch_angle", "Theta0", "validated_score",
    "valid_held_out", "accepted", "acceptance_reason",
]])

ridge_null_draws, pooled_null_summary, null_block_summary = (
    build_blocked_null_diagnostics(logpolar_raw))
assert len(ridge_null_draws) == 96
assert len(null_block_summary) == 12
assert (ridge_null_draws.groupby("m_arms").size() == 32).all()
ridge_geometry_table = ridge_geometry_table.merge(
    pooled_null_summary,
    on="m_arms",
    how="left",
    validate="one_to_one",
)
ridge_geometry_table["null_z"] = (
    (ridge_geometry_table["validated_score"]
     - ridge_geometry_table["null_mean"])
    / ridge_geometry_table["null_std_floor"]
)
ridge_geometry_table["null_role"] = "descriptive_not_model_selection"
assert (
    ridge_geometry_table["null_role"]
    == "descriptive_not_model_selection"
).all()
display(pooled_null_summary)
display(null_block_summary)
display(ridge_geometry_table[[
    "m_arms", "pitch_angle", "validated_score", "valid_held_out",
    "accepted", "acceptance_reason", "null_z", "null_role",
]])
print("RIDGE_NULL_BLOCKS_COMPLETE")

expected_widths = np.array([0.18, 0.22, 0.25, 0.30, 0.35])
assert np.array_equal(RIDGE_WIDTH_SENSITIVITY_KPC, expected_widths)
ridge_width_rows = []
for width_index, width_kpc in enumerate(RIDGE_WIDTH_SENSITIVITY_KPC):
    width_started = perf_counter()
    width_geometry, _, _, width_cache_bytes = conditional_ridge_search(
        logpolar_raw,
        core_width_kpc=float(width_kpc),
        allow_no_acceptance=True,
    )
    assert width_cache_bytes == 0
    assert width_geometry["m_arms"].tolist() == M_COMPARE.tolist()
    for _, geometry in width_geometry.iterrows():
        ridge_width_rows.append({
            "core_width_kpc": float(width_kpc),
            "m_arms": int(geometry["m_arms"]),
            "pitch_angle": float(geometry["pitch_angle"]),
            "Theta0": float(geometry["Theta0"]),
            "validated_score": float(geometry["validated_score"]),
            "full_ridge_score": float(geometry["full_ridge_score"]),
            "phase_stability": float(geometry["phase_stability"]),
            "valid_held_out": int(geometry["valid_held_out"]),
            "pitch_boundary": bool(geometry["pitch_boundary"]),
            "accepted": bool(geometry["accepted"]),
            "acceptance_reason": geometry["acceptance_reason"],
        })
    print(
        "RIDGE_WIDTH_PROGRESS "
        f"width={float(width_kpc):.2f} "
        f"index={width_index + 1}/{len(RIDGE_WIDTH_SENSITIVITY_KPC)} "
        f"elapsed={perf_counter() - width_started:.2f}s"
    )
ridge_width_table = pd.DataFrame(ridge_width_rows)
assert len(ridge_width_table) == 15
assert (
    ridge_width_table.groupby("core_width_kpc").size() == 3).all()
assert (
    ridge_width_table.groupby("m_arms").size()
    == len(RIDGE_WIDTH_SENSITIVITY_KPC)).all()
display(ridge_width_table)
print("RIDGE_WIDTH_M234_COMPLETE")

## 12. Inspect held-sector scores for every selected conditional geometry

The selected m=2, 3, and 4 rows are matched back to the held-sector records for the same arm number and pitch. The plot shows each valid test score at the phase learned only from that fold's training field. Solid lines mark geometries that pass the unchanged ten-sector and positive-score rule; dashed lines mark visible below-threshold conditional geometries, including m=2 when the live evidence remains at nine valid sectors.

This is a fold-level diagnostic rather than a new selector. The accepted flag and reason are inherited from the three frozen conditional rows, and the number of plotted points must equal the sum of their valid-held-sector counts.

In [ ]:
def build_selected_sector_table(ridge_geometry_table, ridge_fold_table):
    """Match selected geometries to valid held-sector rows."""
    output_columns = list(ridge_fold_table.columns)
    for column in [
            "selected_Theta0", "selected_validated_score",
            "accepted", "acceptance_reason"]:
        if column not in output_columns:
            output_columns.append(column)

    selected_frames = []
    for _, geometry in ridge_geometry_table.iterrows():
        selected = ridge_fold_table.loc[
            (ridge_fold_table["m_arms"] == int(geometry["m_arms"]))
            & np.isclose(
                ridge_fold_table["pitch_angle"],
                float(geometry["pitch_angle"]),
            )
            & ridge_fold_table["valid_test"].astype(bool)
        ].copy()
        if len(selected) != int(geometry["valid_held_out"]):
            raise RuntimeError(
                "Selected fold-row count does not match valid_held_out")
        selected["selected_Theta0"] = float(geometry["Theta0"])
        selected["selected_validated_score"] = float(
            geometry["validated_score"])
        selected["accepted"] = bool(geometry["accepted"])
        selected["acceptance_reason"] = geometry[
            "acceptance_reason"]
        if not selected.empty:
            selected_frames.append(selected)

    expected_fold_rows = int(
        ridge_geometry_table["valid_held_out"].sum())
    if selected_frames:
        ridge_sector_table = pd.concat(
            selected_frames, ignore_index=True)
        ridge_sector_table = ridge_sector_table.reindex(
            columns=output_columns)
    else:
        ridge_sector_table = pd.DataFrame(columns=output_columns)
    if len(ridge_sector_table) != expected_fold_rows:
        raise RuntimeError(
            "Selected sector table does not match expected fold rows")
    return ridge_sector_table, expected_fold_rows


ridge_sector_table, expected_fold_rows = build_selected_sector_table(
    ridge_geometry_table, ridge_fold_table)
assert len(ridge_sector_table) == expected_fold_rows
accepted_geometry = ridge_geometry_table["accepted"].astype(bool)
assert (
    ridge_geometry_table.loc[accepted_geometry, "valid_held_out"]
    >= RIDGE_MIN_HELD_OUT_SECTORS
).all()
assert (
    ridge_geometry_table.loc[accepted_geometry, "validated_score"] > 0.0
).all()
rejected_geometry = ridge_geometry_table.loc[~accepted_geometry]
assert (
    rejected_geometry["acceptance_reason"]
    .str.startswith("below_threshold:")
).all()
if len(rejected_geometry):
    assert (~ridge_sector_table.loc[
        ridge_sector_table["m_arms"].isin(
            rejected_geometry["m_arms"]),
        "accepted",
    ]).all()

display(ridge_sector_table[[
    "m_arms", "pitch_angle", "held_sector", "train_Theta0",
    "test_score", "test_radial_fraction", "accepted",
    "acceptance_reason",
]])

fig, ax = plt.subplots(
    figsize=(9.2, 5.0), constrained_layout=True)
mode_colors = {2: "tab:blue", 3: "tab:orange", 4: "tab:green"}
for _, geometry in ridge_geometry_table.iterrows():
    m_arms = int(geometry["m_arms"])
    selected = ridge_sector_table.loc[
        ridge_sector_table["m_arms"] == m_arms
    ].sort_values("held_sector")
    accepted = bool(geometry["accepted"])
    status_label = "accepted" if accepted else "below threshold"
    ax.plot(
        selected["held_sector"],
        selected["test_score"],
        marker="o",
        linestyle="-" if accepted else "--",
        color=mode_colors[m_arms],
        label=f"m={m_arms}: {status_label}",
    )
ax.axhline(0.0, color="black", linewidth=1.0, alpha=0.7)
ax.set(
    xlabel="held azimuth-sector index",
    ylabel="test ridge score at train-selected phase",
    title="Leakage-controlled held-sector scores by conditional geometry",
)
ax.set_xticks(np.arange(RIDGE_N_SECTORS))
ax.legend(title="conditional status")
plt.show()
print("RIDGE_SECTOR_M234_COMPLETE")
print("RIDGE_M234_REAL_COMPLETE")

conditional_fits = {}
conditional_fit_tables = {}
profile_maxima_by_m = {}
for _, geometry_row in ridge_geometry_table.iterrows():
    geometry = geometry_row.to_dict()
    m_arms = int(geometry["m_arms"])
    model = fit_ktz_profile_fixed_geometry(
        fit_table, geometry, radial_fit)
    assert model["m_arms"] == m_arms
    assert model["pitch_angle"] == float(geometry["pitch_angle"])
    assert model["Theta0"] == float(geometry["Theta0"])
    assert model["geometry_source"] == geometry["geometry_source"]
    assert model["ridge_accepted"] == bool(geometry["accepted"])
    assert (model["ridge_acceptance_reason"]
            == geometry["acceptance_reason"])
    model["gradient_dex_per_kpc"] = float(
        -1.0 / (np.log(10.0) * model["h_R"]))

    radius = fit_table["radius_kpc"].to_numpy(float)
    azimuth = fit_table["azimuth_rad"].to_numpy(float)
    theta = spiral_phase(
        radius, azimuth, model["m_arms"],
        model["pitch_angle"], model["Theta0"])
    rate_factor = 1.0 + model["eta"] * arm_profile(
        theta, model["harmonic_g"], model["harmonic_alpha"])
    if not np.isfinite(rate_factor).all() or np.any(rate_factor <= 0.0):
        raise RuntimeError(
            f"Non-positive fitted rate factor for m={m_arms}")
    model_sfr = (
        model["lambda0_0"] * np.exp(-radius / model["h_R"])
        * rate_factor)
    observed_sfr = fit_table["sfr_linear"].to_numpy(float)
    if not np.isfinite(model_sfr).all() or np.any(model_sfr <= 0.0):
        raise RuntimeError(f"Invalid fitted SFR model for m={m_arms}")
    residual_dex = np.log10(observed_sfr) - np.log10(model_sfr)
    conditional_fit_tables[m_arms] = pd.DataFrame({
        "radius_kpc": radius,
        "azimuth_rad": azimuth,
        "sfr_linear": observed_sfr,
        "area_pix": fit_table["area_pix"].to_numpy(float),
        "theta": theta,
        "rate_factor": rate_factor,
        "model_sfr": model_sfr,
        "residual_dex": residual_dex,
    })
    maxima = enumerate_profile_maxima(model)
    maxima.insert(0, "m_arms", m_arms)
    if maxima.empty:
        raise RuntimeError(f"No source-profile maximum for m={m_arms}")
    conditional_fits[m_arms] = model
    profile_maxima_by_m[m_arms] = maxima

assert list(conditional_fits) == M_COMPARE.tolist()
assert list(conditional_fit_tables) == M_COMPARE.tolist()
assert list(profile_maxima_by_m) == M_COMPARE.tolist()
for m_arms in M_COMPARE:
    geometry = ridge_geometry_table.loc[
        ridge_geometry_table["m_arms"] == int(m_arms)].iloc[0]
    model = conditional_fits[int(m_arms)]
    profile_table = conditional_fit_tables[int(m_arms)]
    maxima = profile_maxima_by_m[int(m_arms)]
    assert model["m_arms"] == int(m_arms)
    assert model["pitch_angle"] == float(geometry["pitch_angle"])
    assert model["Theta0"] == float(geometry["Theta0"])
    assert model["ridge_accepted"] == bool(geometry["accepted"])
    assert (model["ridge_acceptance_reason"]
            == geometry["acceptance_reason"])
    assert model["minimum_rate_factor"] > 0.0
    assert (profile_table["rate_factor"] > 0.0).all()
    assert not maxima.empty
assert not conditional_fits[2]["ridge_accepted"]
assert conditional_fits[2]["ridge_acceptance_reason"].startswith(
    "below_threshold:")
print("KTZ_M234_PROFILE_FITS_COMPLETE")
print("HARMONIC_M234_MAXIMA_COMPLETE")
print("POSITIVITY_CHECK_PASS")

## 13. Fixed-geometry KTZ model diagnostics will follow

The plots in this section previously depended on the removed full-field selector and an explicitly retained opposite-winding model. Task 5 will construct the KTZ-compatible source profile at the already frozen ridge geometry and then add projected skeleton, model, and residual diagnostics derived from that fixed fit. This placeholder prevents stale plots from being mistaken for results of the new selector.

In [ ]:
print("FIXED_GEOMETRY_DIAGNOSTICS_PENDING_TASK_5")

## 14. Fixed-geometry spiral skeleton and arm profile will follow

The former skeleton and arm-profile panel used variables produced by the deleted global optimizer. After Task 5 fits only the source profile at fixed ridge geometry, it will rebuild these panels from that result and distinguish the data-derived ridge skeleton from source-model maxima. This executable placeholder intentionally presents no geometry or profile before those dependencies exist.

In [ ]:
print("FIXED_GEOMETRY_DIAGNOSTICS_PENDING_TASK_5")

## 15. Fixed-geometry KTZ-compatible parameter table will follow

The previous table reported intervals from the deleted full-model sector bootstrap. Task 5 will provide the source-profile parameters conditional on the frozen ridge geometry and assemble their diagnostics and stability summaries into the final table. This executable placeholder avoids carrying obsolete bootstrap quantities forward while preserving the notebook's documented cell order for the subsequent implementation.

In [ ]:
print("FIXED_GEOMETRY_PARAMETER_TABLE_PENDING_TASK_5")

## 16. Interpretation limits and the next KTZ step

The fitted parameters describe the best global KTZ-compatible source geometry found in all finite HII SFR bins under the fixed centre, distance, inclination, position angle, and adaptive-bin mask. There is no artificial central hole and no outer-radius quantile cut. One centroid per adaptive bin is used to avoid pseudo-replication, while the bin's member-pixel area ensures that all 681,856 valid HII pixels contribute to the spatial weighting.

`m_arms` is the base global mode, not a catalogue count of visible arm segments. Harmonics of a base `m=1` phase add phase-locked spatial `m=2` and `m=3` power. The candidate table, opposite-winding overlay, skeleton, and residual panels must therefore be interpreted together: a slightly lower global objective does not imply that every asymmetric branch is captured by one pitch angle or that the winding direction is uniquely known. Bootstrap variation across sectors measures sensitivity to which parts of this disturbed disc are sampled.

These parameters can now define an external source-field template for a later oxygen-abundance correlation forward model. They do **not** measure `kappa`, `x0`, `t_star`, enrichment delay, source clustering, arm pattern speed, or arm lifetime. A metallicity analysis should keep the SFR-derived geometry fixed or propagate its bootstrap uncertainty, apply the real oxygen mask and binning to every mock, and compare homogeneous KT18, clustered KT18, and spiral-modulated KTZ models.
